# Prescriptively Unsafe: A Reproducible Audit of When Accurate ICU Prediction Models Would Harm Patients If Used to Guide Treatment

## Notebook 2 of 3: Prediction Instruments, Causal Engine, and the Prescriptive Safety Audit

**What this notebook is.** This is the scientific core of the paper. It does not propose another mortality model and it does not chase discrimination. It builds and validates a deployment evaluation, the **Prescriptive Safety Audit (PSA)**, that answers a question ordinary model reporting never asks: if clinicians acted on a model's implied treatment recommendation, would patients be harmed? A model can be excellently discriminating and well calibrated and still be prescriptively unsafe, because treatment lies on the causal path to the outcome and a risk score deployed as a treatment guide perturbs exactly that path.

**Why prediction accuracy does not imply prescriptive safety.** Consider the deployment behaviour repeatedly documented in critical care: low predicted risk invites de-escalation of aggressive support. If a patient's low predicted risk is itself a consequence of the aggressive support they received, then a score that keys on that signal will recommend withdrawing the very treatment keeping them alive. Discrimination rewards the model for encoding this association; prescriptive safety penalises it. The audit makes that tension measurable.

**The three layers of this notebook.**

1. **Prediction instruments.** Several complementary classifiers (regularised logistic regression, random forest, gradient boosting, and, where available, LightGBM, XGBoost, and CatBoost) are trained purely on pre-anchor covariates, calibrated, and validated internally on held-out MIMIC-IV and externally on eICU. These models are instruments the audit interrogates, not the contribution.

2. **Causal engine.** Kept strictly separate from prediction. A propensity model with overlap and positivity diagnostics; inverse probability weighting with stabilised weights; a marginal structural model; the parametric g-formula; and a doubly robust augmented estimator, cross-fitted. These recover the conditional average treatment effect of post-anchor organ support on in-hospital mortality, which is the object a risk score would unknowingly move.

3. **The Prescriptive Safety Audit.** A small, mathematically defined family of metrics, each answering one distinct deployment question, with bootstrap confidence intervals, external validation, subgroup fairness, and stress testing. Every quantity is defined formally before it is computed.

**Editorial conventions.** British spelling, no em dashes, exact numerical anchoring, and the word audit in preference to framework where either would serve. Every figure is written to disk at 300 dpi as PNG and vector PDF. All expensive steps checkpoint and resume.


In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================
from pathlib import Path

class CFG:
    # --- Inputs: the analytic tables written by Notebook 1 ---
    NB1_DIR       = Path("./nb1_outputs")
    MIMIC_PARQUET = NB1_DIR / "mimic_analytic.parquet"
    EICU_PARQUET  = NB1_DIR / "eicu_analytic.parquet"
    MANIFEST      = NB1_DIR / "nb1_manifest.json"

    # --- Output tree ---
    OUTPUT_DIR = Path("./nb2_outputs")
    CACHE_DIR  = OUTPUT_DIR / "cache"
    MODEL_DIR  = OUTPUT_DIR / "models"
    FIG_DIR    = OUTPUT_DIR / "figures"
    TABLE_DIR  = OUTPUT_DIR / "tables"

    # --- Target and treatment definitions ---
    OUTCOME    = "in_hospital_mortality"     # Y = 1 denotes death (the harm)
    TREATMENT  = "any_organ_support_post"    # A = the post-anchor action a score would move
    TREATMENT_ALTS = ["vaso_post", "vent_post", "rrt_post"]   # sensitivity definitions

    # --- Prescriptive audit parameters ---
    BENEFIT_DELTA = 0.01     # minimum absolute mortality reduction counted as clinically real
    THRESHOLDS    = None     # set in code to a linspace over risk thresholds
    N_THRESHOLD   = 41

    # --- Estimation controls ---
    RANDOM_SEED   = 20260707
    TEST_SIZE     = 0.30     # MIMIC internal hold-out fraction
    CV_FOLDS      = 5
    N_BOOTSTRAP   = 300      # bootstrap replicates for confidence intervals
    CROSSFIT_FOLDS = 5       # cross-fitting folds for CATE and doubly robust estimation
    N_JOBS        = -1
    FAST          = True     # small hyperparameter budgets; set False for full search
    SHAP_SAMPLE   = 400      # rows sampled for SHAP to bound runtime
    FORCE_RECOMPUTE = False

    # --- Figures ---
    FIG_DPI = 300

for d in (CFG.OUTPUT_DIR, CFG.CACHE_DIR, CFG.MODEL_DIR, CFG.FIG_DIR, CFG.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Notebook 2 outputs ->", CFG.OUTPUT_DIR.resolve())
print("Outcome:", CFG.OUTCOME, "| Treatment:", CFG.TREATMENT)


Notebook 2 outputs -> C:\Users\<user>\Downloads\nb2_outputs
Outcome: in_hospital_mortality | Treatment: any_organ_support_post


In [2]:
# =============================================================================
# ENVIRONMENT, LOGGING, REPRODUCIBILITY, OPTIONAL LIBRARIES
# =============================================================================
import os, sys, gc, json, time, logging, platform, warnings, random, itertools
from contextlib import contextmanager
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
try:
    from matplotlib.sankey import Sankey
    _HAS_SANKEY = True
except Exception:
    _HAS_SANKEY = False

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_predict,
                                     RandomizedSearchCV)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             matthews_corrcoef, roc_curve, precision_recall_curve,
                             log_loss)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

try:
    from scipy import stats as sstats
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _try(name):
    try:
        return __import__(name), True
    except Exception:
        return None, False

lgb, _HAS_LGB = _try("lightgbm")
xgb, _HAS_XGB = _try("xgboost")
_cat_mod, _HAS_CAT = _try("catboost")
shap, _HAS_SHAP = _try("shap")

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
os.environ["PYTHONHASHSEED"] = str(CFG.RANDOM_SEED)
random.seed(CFG.RANDOM_SEED); np.random.seed(CFG.RANDOM_SEED)

LOG_PATH = CFG.OUTPUT_DIR / "nb2_run.log"
logger = logging.getLogger("psa"); logger.setLevel(logging.INFO); logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(_fmt); logger.addHandler(_sh)
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8"); _fh.setFormatter(_fmt); logger.addHandler(_fh)
logger.propagate = False

@contextmanager
def timed(label):
    t = time.time(); logger.info("START  %s", label)
    try:
        yield
    finally:
        logger.info("DONE   %s  (%.1fs)", label, time.time() - t)

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(CFG.FIG_DIR / f"{name}.{ext}", dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close(fig); logger.info("figure -> %s", name)

def cache_path(name, ext="parquet"): return CFG.CACHE_DIR / f"{name}.{ext}"
def cache_ok(name, ext="parquet"): return cache_path(name, ext).exists() and not CFG.FORCE_RECOMPUTE

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CFG.FIG_DPI,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
PALETTE = {"MIMIC": "#3b76af", "eICU": "#5aa469", "harm": "#c0392b", "safe": "#2e8b57",
           "neutral": "#7f8c8d", "accent": "#8e44ad"}

logger.info("Python %s | numpy %s | pandas %s | sklearn %s",
            sys.version.split()[0], np.__version__, pd.__version__, __import__("sklearn").__version__)
logger.info("optional: lightgbm=%s xgboost=%s catboost=%s shap=%s scipy=%s",
            _HAS_LGB, _HAS_XGB, _HAS_CAT, _HAS_SHAP, _HAS_SCIPY)
RUN = {"started": datetime.now().isoformat(timespec="seconds"), "seed": CFG.RANDOM_SEED}


C:\Users\<user>\anaconda3-v2\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


00:08:17 | INFO    | Python 3.11.5 | numpy 1.24.4 | pandas 2.3.3 | sklearn 1.3.0
00:08:17 | INFO    | optional: lightgbm=True xgboost=True catboost=False shap=False scipy=True


In [3]:
# =============================================================================
# LOAD NOTEBOOK 1 ARTEFACTS AND DEFINE VARIABLE ROLES
# =============================================================================
with timed("load analytic tables"):
    mimic = pd.read_parquet(CFG.MIMIC_PARQUET)
    eicu  = pd.read_parquet(CFG.EICU_PARQUET)
    manifest = json.load(open(CFG.MANIFEST)) if CFG.MANIFEST.exists() else {}

assert list(mimic.columns) == list(eicu.columns), "MIMIC and eICU schemas differ; re-run Notebook 1."
logger.info("MIMIC %d rows | eICU %d rows | %d columns", len(mimic), len(eicu), mimic.shape[1])

# Column roles. Features are strictly pre-anchor; treatment and outcomes are excluded
# from the covariate set so nothing on the causal path leaks into the risk model.
ID_COLS      = ["db", "patient_id", "hadm_id"]
OUTCOME_COLS = ["in_hospital_mortality", "icu_mortality", "mortality_28d", "hosp_los_days"]
POST_COLS    = ["los_h"]   # total ICU stay is realised after the anchor: not a feature
TREAT_COLS   = ["vaso_post", "vaso_day1", "inotrope_post", "vent_post", "vent_day1",
                "invasive_vent_post", "rrt_post", "rrt_day1",
                "n_pressor_types_post", "any_organ_support_post"]
CAT_COLS     = ["sex", "race_group", "admission_type", "insurance"]

_exclude = set(ID_COLS + OUTCOME_COLS + POST_COLS + TREAT_COLS + ["sofa_coverage"])
NUM_COLS = [c for c in mimic.columns
            if c not in _exclude and c not in CAT_COLS
            and pd.api.types.is_numeric_dtype(mimic[c])]
FEATURES = NUM_COLS + CAT_COLS

Y_COL, A_COL = CFG.OUTCOME, CFG.TREATMENT
logger.info("Design: %d numeric + %d categorical = %d covariates", len(NUM_COLS), len(CAT_COLS), len(FEATURES))
logger.info("MIMIC outcome prevalence %.3f | treatment prevalence %.3f",
            mimic[Y_COL].mean(), mimic[A_COL].mean())
logger.info("eICU  outcome prevalence %.3f | treatment prevalence %.3f",
            eicu[Y_COL].mean(), eicu[A_COL].mean())
pd.DataFrame({"role": ["features", "outcome", "treatment", "excluded"],
              "n": [len(FEATURES), 1, 1, len(_exclude)]})


00:08:17 | INFO    | START  load analytic tables
00:08:17 | INFO    | DONE   load analytic tables  (0.2s)
00:08:17 | INFO    | MIMIC 36069 rows | eICU 86403 rows | 196 columns
00:08:17 | INFO    | Design: 173 numeric + 4 categorical = 177 covariates
00:08:17 | INFO    | MIMIC outcome prevalence 0.106 | treatment prevalence 0.374
00:08:17 | INFO    | eICU  outcome prevalence 0.099 | treatment prevalence 0.457


,role,n
0,features,177
1,outcome,1
2,treatment,1
3,excluded,19


---
## 1. Notation, causal setup, and the estimands

We fix notation once and use it throughout. For patient $i$ evaluated at the landmark anchor $t_0 + 24\,\text{h}$:

- $L_i \in \mathbb{R}^p$ are the pre-anchor covariates (physiology, laboratory aggregates, first-day SOFA, comorbidity burden, demographics). By the landmark construction in Notebook 1, $L_i$ is measured strictly before the anchor.
- $A_i \in \{0,1\}$ is the post-anchor treatment (organ support). It is measured strictly after the anchor, so it cannot enter $L_i$.
- $Y_i \in \{0,1\}$ is in-hospital mortality, $Y_i = 1$ denoting death.

**Prediction object.** A deployable risk model $f$ uses only covariates, $r_i = f(L_i) \approx \Pr(Y_i = 1 \mid L_i)$. It is treatment blind by necessity: at decision time the post-anchor action has not occurred.

**Causal object.** Using potential outcomes $Y_i(a)$ under $\text{do}(A_i = a)$, the conditional average treatment effect is
$$\tau(L_i) = \mathbb{E}[Y_i(1) - Y_i(0) \mid L_i].$$
Because death is the harmful outcome, a negative $\tau$ means treatment is beneficial. We define the **benefit** $b_i = -\tau(L_i) \ge 0$ when treatment reduces mortality.

**Identifying assumptions**, stated so a reader can challenge each one.
1. *Consistency.* The observed $Y_i$ equals $Y_i(A_i)$.
2. *Conditional exchangeability.* $Y_i(a) \perp A_i \mid L_i$: no unmeasured confounding given the recorded pre-anchor covariates. This is not testable; we probe its fragility with an E-value.
3. *Positivity.* $0 < \Pr(A_i = 1 \mid L_i) < 1$ across the covariate support. We test this directly with overlap diagnostics and report trimming.

Under these, $\tau(L)$ is identified by $\mathbb{E}[Y \mid A=1, L] - \mathbb{E}[Y \mid A=0, L]$ and by inverse probability weighting, which we estimate by complementary methods and reconcile.

**The link that makes prediction prescriptive.** A deployed risk score implies an action only once paired with a decision rule. We study the clinically documented **risk-guided de-escalation** rule at threshold $\tau_r$: withhold post-anchor support when $r_i < \tau_r$. Deployment changes care for a patient when the rule's action differs from observed care. The audit asks, over the range of $\tau_r$, how often those induced changes are harmful, how large the harm is, and whether the risk ordering the model actually uses is even aligned with the benefit ordering that should govern treatment. Every metric in Section 4 is a precise answer to one of those questions.


In [4]:
# =============================================================================
# DESIGN MATRIX: imputation, missing-indicators, one-hot encoding
# The transformer is fit on MIMIC training data only and reused everywhere, so
# external and held-out data never inform preprocessing.
# =============================================================================
from sklearn.base import clone

def build_preprocessor():
    num = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_COLS), ("cat", cat, CAT_COLS)],
                             remainder="drop", verbose_feature_names_out=False)

def frame_xya(df):
    X = df[FEATURES].copy()
    y = df[Y_COL].astype(int).values
    a = df[A_COL].astype(int).values
    return X, y, a

# split MIMIC into development and internal hold-out (stratified on the outcome)
mimic_dev, mimic_hold = train_test_split(
    mimic, test_size=CFG.TEST_SIZE, random_state=CFG.RANDOM_SEED, stratify=mimic[Y_COL])
logger.info("MIMIC development %d | internal hold-out %d | eICU external %d",
            len(mimic_dev), len(mimic_hold), len(eicu))

PRE = build_preprocessor()
Xdev_raw, y_dev, a_dev = frame_xya(mimic_dev)
PRE.fit(Xdev_raw)
FEAT_NAMES = list(PRE.get_feature_names_out())

def design(df):
    X, y, a = frame_xya(df)
    return PRE.transform(X), y, a

X_dev,  y_dev,  a_dev  = design(mimic_dev)
X_hold, y_hold, a_hold = design(mimic_hold)
X_ext,  y_ext,  a_ext  = design(eicu)
logger.info("Design matrix width after encoding: %d", X_dev.shape[1])
CFG.THRESHOLDS = np.linspace(0.02, 0.60, CFG.N_THRESHOLD)
print("Encoded feature count:", len(FEAT_NAMES))


00:08:17 | INFO    | MIMIC development 25248 | internal hold-out 10821 | eICU external 86403
00:08:19 | INFO    | Design matrix width after encoding: 324
Encoded feature count: 324


In [5]:
# =============================================================================
# Save design matrices for reuse in NB3 (for transportability, etc.)
# =============================================================================
np.savez(CFG.CACHE_DIR / "design_matrices.npz",
         X_dev=X_dev, X_ext=X_ext, y_dev=y_dev, y_ext=y_ext,
         a_dev=a_dev, a_ext=a_ext, X_hold=X_hold, y_hold=y_hold, a_hold=a_hold)
logger.info("Saved design matrices to cache.")

00:08:19 | INFO    | Saved design matrices to cache.


---
## 2. Prediction instruments

These classifiers are the objects the audit interrogates; the audit, not the classifiers, is the contribution. We include complementary model families so that the prescriptive findings cannot be dismissed as an artefact of one learner.

- **Regularised logistic regression.** A transparent linear baseline whose behaviour under deployment is fully legible, and whose calibrated probabilities are a fair reference.
- **Random forest.** A bagged, low-variance nonlinear learner with a different inductive bias from boosting.
- **Gradient boosting.** The workhorse of tabular clinical prediction. We use LightGBM, XGBoost, and CatBoost where installed, and fall back to scikit-learn histogram gradient boosting otherwise, so the notebook runs anywhere while using the strongest available instrument as the primary model.

Each model is trained on pre-anchor covariates only, then wrapped in post hoc probability calibration. We report discrimination and calibration internally on held-out MIMIC-IV and externally on eICU, with bootstrap confidence intervals and a DeLong test for paired AUROC differences. Nested cross-validation guards the primary model's hyperparameters against optimistic selection.

In [6]:
# =============================================================================
# DeLong paired AUROC test (fast midrank implementation) and metric helpers
# =============================================================================
def _compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def delong_roc_variance(y_true, y_score):
    order = np.argsort(-y_score)
    label = y_true[order]; score = y_score[order]
    pos = score[label == 1]; neg = score[label == 0]
    m, n = len(pos), len(neg)
    tx = _compute_midrank(pos); ty = _compute_midrank(neg); tz = _compute_midrank(np.r_[pos, neg])
    auc = (tz[:m].sum() - m * (m + 1) / 2) / (m * n)
    v01 = (tz[:m] - tx) / n
    v10 = 1 - (tz[m:] - ty) / m
    sx = np.var(v01, ddof=1) / m; sy = np.var(v10, ddof=1) / n
    return auc, sx + sy

def delong_test(y_true, s1, s2):
    """Two-sided paired DeLong test for AUROC(s1) vs AUROC(s2)."""
    order = np.argsort(-np.r_[s1, s2])  # not used; kept for clarity
    y = y_true.astype(int)
    a1, var1 = delong_roc_variance(y, s1)
    a2, var2 = delong_roc_variance(y, s2)
    # covariance via structural components on the shared labels
    def _components(score):
        o = np.argsort(-score); lab = y[o]; sc = score[o]
        pos = sc[lab == 1]; neg = sc[lab == 0]; m, n = len(pos), len(neg)
        tx = _compute_midrank(pos); ty = _compute_midrank(neg); tz = _compute_midrank(np.r_[pos, neg])
        return (tz[:m] - tx) / n, 1 - (tz[m:] - ty) / m
    v01_1, v10_1 = _components(s1); v01_2, v10_2 = _components(s2)
    m = int(y.sum()); n = len(y) - m
    cov = np.cov(np.vstack([v01_1, v01_2]))[0, 1] / m + np.cov(np.vstack([v10_1, v10_2]))[0, 1] / n
    se = np.sqrt(max(var1 + var2 - 2 * cov, 1e-12))
    z = (a1 - a2) / se
    from math import erf, sqrt
    p = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
    return a1, a2, z, p, se

def auroc_ci(y, s, alpha=0.05):
    auc, var = delong_roc_variance(y.astype(int), s)
    from math import sqrt, erf
    z = 1.959963985
    lo, hi = auc - z * sqrt(var), auc + z * sqrt(var)
    return auc, max(0, lo), min(1, hi)

def expected_calibration_error(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); idx = np.digitize(p, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += abs(p[m].mean() - y[m].mean()) * m.mean()
    return ece

def classification_metrics(y, p):
    yhat = (p >= 0.5).astype(int)
    auc, lo, hi = auroc_ci(y, p)
    return {"AUROC": auc, "AUROC_lo": lo, "AUROC_hi": hi,
            "AUPRC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p),
            "ECE": expected_calibration_error(y, p),
            "MCC": matthews_corrcoef(y, yhat),
            "logloss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))}

def bootstrap_ci(fn, *arrays, B=CFG.N_BOOTSTRAP, seed=CFG.RANDOM_SEED, alpha=0.05):
    rng = np.random.default_rng(seed); n = len(arrays[0]); stats = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        try:
            stats.append(fn(*[a[idx] for a in arrays]))
        except Exception:
            continue
    stats = np.array(stats)
    return float(np.mean(stats)), float(np.percentile(stats, 100 * alpha / 2)), float(np.percentile(stats, 100 * (1 - alpha / 2)))

logger.info("DeLong and metric helpers ready.")


00:08:19 | INFO    | DeLong and metric helpers ready.


In [7]:
# =============================================================================
# MODEL ZOO: blind classifiers f(L), plus a boosting instrument factory
# =============================================================================
def boosting_clf(seed=CFG.RANDOM_SEED):
    """Strongest available gradient-boosting classifier; identical role everywhere."""
    if _HAS_LGB:
        return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=31,
                                  subsample=0.8, colsample_bytree=0.8, random_state=seed,
                                  n_jobs=CFG.N_JOBS, verbose=-1)
    if _HAS_XGB:
        return xgb.XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=4,
                                 subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                                 random_state=seed, n_jobs=CFG.N_JOBS, tree_method="hist")
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                          max_depth=None, random_state=seed)

def model_zoo():
    zoo = {
        "logreg": LogisticRegression(penalty="l2", C=1.0, max_iter=2000,
                                     class_weight=None, solver="lbfgs"),
        "random_forest": RandomForestClassifier(n_estimators=400, max_depth=None,
                                                min_samples_leaf=5, n_jobs=CFG.N_JOBS,
                                                random_state=CFG.RANDOM_SEED),
        "hist_gb": HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                                  random_state=CFG.RANDOM_SEED),
    }
    if _HAS_LGB: zoo["lightgbm"] = boosting_clf()
    if _HAS_XGB: zoo["xgboost"] = xgb.XGBClassifier(n_estimators=400, learning_rate=0.03,
                                    max_depth=4, subsample=0.8, colsample_bytree=0.8,
                                    eval_metric="logloss", random_state=CFG.RANDOM_SEED,
                                    n_jobs=CFG.N_JOBS, tree_method="hist")
    if _HAS_CAT: zoo["catboost"] = _cat_mod.CatBoostClassifier(iterations=400, learning_rate=0.03,
                                    depth=4, random_seed=CFG.RANDOM_SEED, verbose=False)
    return zoo

PRIMARY = "lightgbm" if _HAS_LGB else ("xgboost" if _HAS_XGB else "hist_gb")
logger.info("Model zoo: %s | primary instrument: %s", list(model_zoo().keys()), PRIMARY)


00:08:20 | INFO    | Model zoo: ['logreg', 'random_forest', 'hist_gb', 'lightgbm', 'xgboost'] | primary instrument: lightgbm


In [8]:
# =============================================================================
# Nested cross-validation for the primary instrument (honest AUROC)
# Inner loop tunes a small budget; outer loop estimates generalisation AUROC.
# =============================================================================
def nested_cv_auroc(estimator, param_dist, X, y, outer=CFG.CV_FOLDS,
                    inner=3, n_iter=(6 if CFG.FAST else 25)):
    okf = StratifiedKFold(n_splits=outer, shuffle=True, random_state=CFG.RANDOM_SEED)
    scores = []
    for k, (tr, te) in enumerate(okf.split(X, y)):
        ikf = StratifiedKFold(n_splits=inner, shuffle=True, random_state=CFG.RANDOM_SEED + k)
        search = RandomizedSearchCV(clone(estimator), param_dist, n_iter=n_iter, scoring="roc_auc",
                                    cv=ikf, random_state=CFG.RANDOM_SEED + k, n_jobs=CFG.N_JOBS)
        search.fit(X[tr], y[tr])
        p = search.best_estimator_.predict_proba(X[te])[:, 1]
        scores.append(roc_auc_score(y[te], p))
    return np.array(scores)

if cache_ok("nested_cv", "json"):
    nested = json.load(open(cache_path("nested_cv", "json")))
    logger.info("loaded nested CV from cache")
else:
    with timed("nested CV (primary instrument)"):
        if _HAS_LGB:
            base = lgb.LGBMClassifier(random_state=CFG.RANDOM_SEED, n_jobs=CFG.N_JOBS, verbose=-1)
            grid = {"n_estimators": [200, 400, 600], "learning_rate": [0.02, 0.05, 0.1],
                    "num_leaves": [15, 31, 63], "subsample": [0.7, 0.9]}
        else:
            base = HistGradientBoostingClassifier(random_state=CFG.RANDOM_SEED)
            grid = {"max_iter": [200, 400], "learning_rate": [0.03, 0.05, 0.1],
                    "max_leaf_nodes": [15, 31, 63], "l2_regularization": [0.0, 1.0]}
        sc = nested_cv_auroc(base, grid, X_dev, y_dev)
        nested = {"model": PRIMARY, "auroc_mean": float(sc.mean()), "auroc_sd": float(sc.std()),
                  "folds": sc.tolist()}
        json.dump(nested, open(cache_path("nested_cv", "json"), "w"), indent=2)
logger.info("Nested CV AUROC (%s): %.3f +/- %.3f", nested["model"], nested["auroc_mean"], nested["auroc_sd"])
nested


00:08:20 | INFO    | loaded nested CV from cache
00:08:20 | INFO    | Nested CV AUROC (lightgbm): 0.906 +/- 0.005


{'model': 'lightgbm',
 'auroc_mean': 0.9055782339075845,
 'auroc_sd': 0.004814557406168133,
 'folds': [0.9046895562065612,
  0.9080774406655249,
  0.913441349962637,
  0.899802069573787,
  0.9018807531294125]}

In [9]:
# =============================================================================
# Train and calibrate every blind model on MIMIC development data
# Calibration uses an internal cross-validated split so hold-out stays untouched.
# =============================================================================
def calib_method(n_pos):
    return "isotonic" if n_pos >= 1000 else "sigmoid"

models = {}
mdl_cache = CFG.MODEL_DIR / "blind_models.joblib"
if mdl_cache.exists() and not CFG.FORCE_RECOMPUTE:
    models = joblib.load(mdl_cache); logger.info("loaded %d calibrated models from cache", len(models))
else:
    with timed("train + calibrate blind models"):
        cm = calib_method(int(y_dev.sum()))
        for name, est in model_zoo().items():
            clf = CalibratedClassifierCV(est, method=cm, cv=3)
            clf.fit(X_dev, y_dev)
            models[name] = clf
            logger.info("  trained %-14s (calibration=%s)", name, cm)
        joblib.dump(models, mdl_cache)

# store predicted risk on internal hold-out and external eICU for every model
def predict_all(models, X):
    return {name: m.predict_proba(X)[:, 1] for name, m in models.items()}

risk_hold = predict_all(models, X_hold)
risk_ext  = predict_all(models, X_ext)
# the primary risk score the audit will centre on
r_hold = risk_hold[PRIMARY]; r_ext = risk_ext[PRIMARY]
logger.info("Predicted risk ready (primary=%s): hold mean %.3f | ext mean %.3f",
            PRIMARY, r_hold.mean(), r_ext.mean())


00:08:22 | INFO    | loaded 5 calibrated models from cache


  File "C:\Users\<user>\anaconda3-v2\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\<user>\anaconda3-v2\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


00:08:31 | INFO    | Predicted risk ready (primary=lightgbm): hold mean 0.107 | ext mean 0.101


In [10]:
# =============================================================================
# Discrimination and calibration: internal hold-out and external eICU
# =============================================================================
rows = []
for name in models:
    for coh, (y_, p_) in {"MIMIC-hold": (y_hold, risk_hold[name]),
                          "eICU-ext":   (y_ext,  risk_ext[name])}.items():
        m = classification_metrics(y_, p_)
        m.update({"model": name, "cohort": coh}); rows.append(m)
perf = pd.DataFrame(rows)[["model", "cohort", "AUROC", "AUROC_lo", "AUROC_hi",
                           "AUPRC", "Brier", "ECE", "MCC", "logloss"]].round(4)
perf.to_csv(CFG.TABLE_DIR / "model_performance.csv", index=False)

# paired DeLong: every model vs the primary, on external data
del_rows = []
for name in models:
    if name == PRIMARY:
        continue
    a1, a2, z, p, se = delong_test(y_ext, risk_ext[PRIMARY], risk_ext[name])
    p_str = "<0.001" if p < 0.001 else f"{p:.4f}"
    del_rows.append({"comparison": f"{PRIMARY} vs {name}", "cohort": "eICU-ext",
                 "AUROC_primary": round(a1, 4), "AUROC_other": round(a2, 4),
                 "z": round(z, 3), "p_value": p_str, "se": round(se, 4)})
delong_df = pd.DataFrame(del_rows)
delong_df.to_csv(CFG.TABLE_DIR / "delong_tests.csv", index=False)
logger.info("Performance and DeLong tables written.")
perf


00:08:34 | INFO    | Performance and DeLong tables written.


,model,cohort,AUROC,AUROC_lo,AUROC_hi,AUPRC,Brier,ECE,MCC,logloss
0,logreg,MIMIC-hold,0.8932,0.8845,0.9019,0.5257,0.0676,0.0081,0.3867,0.2237
1,logreg,eICU-ext,0.8132,0.8087,0.8178,0.3680,0.0771,0.0298,0.2864,0.2705
2,random_forest,MIMIC-hold,0.8906,0.8819,0.8993,0.5233,0.0679,0.0071,0.3905,0.2263
3,random_forest,eICU-ext,0.8088,0.8043,0.8133,0.3544,0.0802,0.0334,0.2913,0.2731
4,hist_gb,MIMIC-hold,0.9055,0.8976,0.9134,0.5568,0.0647,0.0082,0.4253,0.2132
5,hist_gb,eICU-ext,0.8159,0.8114,0.8203,0.3666,0.0787,0.0401,0.2918,0.2752
6,lightgbm,MIMIC-hold,0.9088,0.9011,0.9165,0.5688,0.0639,0.0079,0.4375,0.2100
7,lightgbm,eICU-ext,0.8218,0.8174,0.8261,0.3806,0.0775,0.0368,0.3018,0.2686
8,xgboost,MIMIC-hold,0.9072,0.8992,0.9151,0.5614,0.0643,0.0055,0.4308,0.2121
9,xgboost,eICU-ext,0.8206,0.8162,0.8249,0.3775,0.0776,0.0383,0.2965,0.2710


In [11]:
# =============================================================================
# Figures: ROC, precision-recall, and calibration (internal vs external)
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for name in models:
    fpr, tpr, _ = roc_curve(y_hold, risk_hold[name])
    axes[0].plot(fpr, tpr, lw=1.4, label=f"{name} ({roc_auc_score(y_hold, risk_hold[name]):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8); axes[0].set_title("ROC (MIMIC hold-out)")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].legend(fontsize=7)

for name in models:
    pr, rc, _ = precision_recall_curve(y_ext, risk_ext[name])
    axes[1].plot(rc, pr, lw=1.4, label=f"{name} ({average_precision_score(y_ext, risk_ext[name]):.3f})")
axes[1].axhline(y_ext.mean(), color="k", ls="--", lw=0.8)
axes[1].set_title("Precision-recall (eICU external)")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend(fontsize=7)

for coh, y_, p_, c in [("MIMIC hold", y_hold, r_hold, PALETTE["MIMIC"]),
                       ("eICU ext", y_ext, r_ext, PALETTE["eICU"])]:
    frac, mean_pred = calibration_curve(y_, p_, n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac, "o-", color=c, lw=1.5, label=f"{coh} (ECE {expected_calibration_error(y_, p_):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[2].set_title(f"Calibration of primary model ({PRIMARY})")
axes[2].set_xlabel("mean predicted risk"); axes[2].set_ylabel("observed frequency"); axes[2].legend(fontsize=8)
savefig(fig, "fig_discrimination_calibration")
print("Discrimination and calibration figure written.")


00:08:36 | INFO    | figure -> fig_discrimination_calibration
Discrimination and calibration figure written.


---
## 3. Causal engine

This section is deliberately separate from prediction. Its job is to estimate the causal effect of post-anchor organ support on in-hospital mortality, both marginally and as a function of covariates, because that effect is the quantity a deployed risk score would move without knowing it.

We estimate the average treatment effect four ways whose biases differ: a stabilised inverse probability weighted contrast, a marginal structural model, the parametric g-formula by outcome standardisation, and an augmented (doubly robust) estimator that is consistent if either the propensity or the outcome model is correct. Agreement across methods is evidence of robustness; disagreement is itself informative. Before any effect is trusted we test positivity through propensity overlap, and after estimation we quantify how strong an unmeasured confounder would need to be to overturn the result via the E-value.

In [12]:
# =============================================================================
# Propensity model and positivity / overlap diagnostics
# =============================================================================
with timed("propensity model"):
    ps_model = CalibratedClassifierCV(boosting_clf(), method="isotonic", cv=3)
    ps_model.fit(X_dev, a_dev)

def propensity(X):
    return np.clip(ps_model.predict_proba(X)[:, 1], 1e-3, 1 - 1e-3)

ps_dev  = propensity(X_dev)
ps_hold = propensity(X_hold)
ps_ext  = propensity(X_ext)

# overlap diagnostics on development data
def overlap_report(ps, a, eps=0.02):
    treated, control = ps[a == 1], ps[a == 0]
    frac_extreme = np.mean((ps < eps) | (ps > 1 - eps))
    common_lo = max(treated.min(), control.min()); common_hi = min(treated.max(), control.max())
    outside = np.mean((ps < common_lo) | (ps > common_hi))
    return {"n": len(ps), "treated_frac": float(a.mean()),
            "frac_extreme_ps": float(frac_extreme),
            "common_support": [float(common_lo), float(common_hi)],
            "frac_outside_common_support": float(outside)}

ovl = overlap_report(ps_dev, a_dev)
json.dump(ovl, open(CFG.TABLE_DIR / "positivity_overlap.json", "w"), indent=2)
logger.info("Positivity: %.1f%% extreme propensity; common support [%.2f, %.2f]",
            100 * ovl["frac_extreme_ps"], *ovl["common_support"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].hist(ps_dev[a_dev == 1], bins=30, alpha=0.6, color=PALETTE["harm"], label="treated", density=True)
ax[0].hist(ps_dev[a_dev == 0], bins=30, alpha=0.6, color=PALETTE["MIMIC"], label="control", density=True)
ax[0].set_title("Propensity overlap (MIMIC dev)"); ax[0].set_xlabel("P(A=1 | L)"); ax[0].legend()
# mirrored density
from numpy import histogram
b = np.linspace(0, 1, 31)
h1, _ = histogram(ps_dev[a_dev == 1], bins=b, density=True)
h0, _ = histogram(ps_dev[a_dev == 0], bins=b, density=True)
ctr = 0.5 * (b[1:] + b[:-1])
ax[1].barh(ctr, h1, height=1/30, color=PALETTE["harm"], alpha=0.7, label="treated")
ax[1].barh(ctr, -h0, height=1/30, color=PALETTE["MIMIC"], alpha=0.7, label="control")
ax[1].axvline(0, color="k", lw=0.6); ax[1].set_title("Mirrored propensity density")
ax[1].set_ylabel("P(A=1 | L)"); ax[1].set_xlabel("density (treated right, control left)"); ax[1].legend()
savefig(fig, "fig_propensity_overlap")
ovl


00:08:36 | INFO    | START  propensity model
00:08:42 | INFO    | DONE   propensity model  (5.6s)
00:08:43 | INFO    | Positivity: 15.5% extreme propensity; common support [0.02, 0.91]
00:08:44 | INFO    | figure -> fig_propensity_overlap


{'n': 25248,
 'treated_frac': 0.37507921419518375,
 'frac_extreme_ps': 0.15466571609632446,
 'common_support': [0.021511221430932394, 0.9076624698696496],
 'frac_outside_common_support': 0.2525744613434727}

In [13]:
# =============================================================================
# Stabilised inverse probability weights and weight diagnostics
# =============================================================================
def stabilised_weights(a, ps, p_treat):
    return np.where(a == 1, p_treat / ps, (1 - p_treat) / (1 - ps))

p_treat_dev = a_dev.mean()
sw_dev  = stabilised_weights(a_dev,  ps_dev,  p_treat_dev)
sw_hold = stabilised_weights(a_hold, ps_hold, p_treat_dev)
sw_ext  = stabilised_weights(a_ext,  ps_ext,  eicu[A_COL].mean())

# truncate at the 1st/99th percentile to tame extreme weights (reported)
def truncate(w, lo=1, hi=99):
    a_, b_ = np.percentile(w, [lo, hi]); return np.clip(w, a_, b_)
sw_dev_t = truncate(sw_dev)
logger.info("Stabilised weights (dev): mean %.3f (target 1.0), max %.1f -> truncated max %.1f",
            sw_dev.mean(), sw_dev.max(), sw_dev_t.max())

# weighted standardised mean differences (balance) for a few key covariates
def smd(x, a, w=None):
    if w is None: w = np.ones_like(a, dtype=float)
    def wm(v, m): ww = w[m]; return np.average(v[m], weights=ww)
    def wv(v, m): ww = w[m]; mu = wm(v, m); return np.average((v[m]-mu)**2, weights=ww)
    m1, m0 = a == 1, a == 0
    num = wm(x, m1) - wm(x, m0); den = np.sqrt(0.5 * (wv(x, m1) + wv(x, m0)) + 1e-12)
    return num / den

key_idx = [FEAT_NAMES.index(c) for c in ["sofa_total", "admission_age", "lactate_max", "creatinine_max"]
           if c in FEAT_NAMES]
bal_rows = []
for j in key_idx:
    xj = X_dev[:, j]
    bal_rows.append({"covariate": FEAT_NAMES[j],
                     "SMD_unweighted": round(smd(xj, a_dev), 3),
                     "SMD_weighted": round(smd(xj, a_dev, sw_dev_t), 3)})
balance = pd.DataFrame(bal_rows)
balance.to_csv(CFG.TABLE_DIR / "covariate_balance.csv", index=False)
balance


00:08:44 | INFO    | Stabilised weights (dev): mean 0.708 (target 1.0), max 17.4 -> truncated max 2.5


,covariate,SMD_unweighted,SMD_weighted
0,sofa_total,1.195,0.680
1,admission_age,0.004,0.006
2,lactate_max,0.413,0.280
3,creatinine_max,0.327,0.207


In [14]:
# =============================================================================
# Four estimators of the average treatment effect, cross-fitted on all MIMIC
# ATE is a risk difference in mortality: negative means treatment saves lives.
# =============================================================================
X_all, y_all, a_all = design(mimic)

def crossfit_ate(X, y, a, K=CFG.CROSSFIT_FOLDS, seed=CFG.RANDOM_SEED):
    kf = StratifiedKFold(n_splits=K, shuffle=True, random_state=seed)
    mu1 = np.zeros(len(y)); mu0 = np.zeros(len(y)); e = np.zeros(len(y))
    for tr, te in kf.split(X, y):
        # propensity
        pm = boosting_clf(); pm.fit(X[tr], a[tr]); e[te] = np.clip(pm.predict_proba(X[te])[:, 1], 1e-3, 1-1e-3)
        # outcome models by arm (T-learner)
        tr1 = tr[a[tr] == 1]; tr0 = tr[a[tr] == 0]
        m1 = boosting_clf(); m1.fit(X[tr1], y[tr1]); mu1[te] = m1.predict_proba(X[te])[:, 1]
        m0 = boosting_clf(); m0.fit(X[tr0], y[tr0]); mu0[te] = m0.predict_proba(X[te])[:, 1]
    return mu1, mu0, e

if cache_ok("crossfit", "npz"):
    d = np.load(cache_path("crossfit", "npz")); mu1, mu0, e_all = d["mu1"], d["mu0"], d["e"]
    logger.info("loaded cross-fitted nuisance estimates from cache")
else:
    with timed("cross-fitted nuisance estimation"):
        mu1, mu0, e_all = crossfit_ate(X_all, y_all, a_all)
        np.savez(cache_path("crossfit", "npz"), mu1=mu1, mu0=mu0, e=e_all)

pt = a_all.mean()
e  = np.clip(e_all, 1e-3, 1 - 1e-3)

# --- common-support trim (report what is dropped) ---
lo = max(e[a_all == 1].min(), e[a_all == 0].min())
hi = min(e[a_all == 1].max(), e[a_all == 0].max())
keep = (e >= lo) & (e <= hi)
frac_dropped = float(1 - keep.mean())

# --- overlap weights: the exact-balance property holds only for a LOGISTIC MLE
#     propensity, so estimate a dedicated logistic PS here. The boosted `e` above
#     is kept for the common-support trim and the AIPW/g-formula estimators. ---
from sklearn.linear_model import LogisticRegression as _LR
_ps_ow = _LR(max_iter=5000, C=1.0)
_ps_ow.fit(StandardScaler().fit_transform(X_all), a_all)
e_ow = np.clip(_ps_ow.predict_proba(StandardScaler().fit_transform(X_all))[:, 1], 1e-3, 1-1e-3)
ow = np.where(a_all == 1, 1 - e_ow, e_ow)

def _ato(y, a, e, ow, mu1, mu0, idx=None):
    if idx is None: idx = np.arange(len(y))
    a_, y_, e_, ow_ = a[idx], y[idx], e[idx], ow[idx]
    m1_, m0_ = mu1[idx], mu0[idx]
    w1, w0 = a_ * (1 - e_), (1 - a_) * e_
    ipw = np.sum(w1 * y_) / np.sum(w1) - np.sum(w0 * y_) / np.sum(w0)
    g   = np.sum(ow_ * (m1_ - m0_)) / np.sum(ow_)
    psi = ow_ * ((m1_ - m0_) + a_ * (y_ - m1_) / e_ - (1 - a_) * (y_ - m0_) / (1 - e_))
    dr  = np.sum(psi) / np.sum(ow_)
    return ipw, g, dr

ate_ato_ipw, ate_ato_g, ate_ato_dr = _ato(y_all, a_all, e_ow, ow, mu1, mu0)

# original four estimators, now restricted to common support (report both estimands honestly)
kt = keep
ate_g   = float(np.mean((mu1 - mu0)[kt]))
sw_all  = stabilised_weights(a_all, e, pt)
ate_ipw = float(np.sum(sw_all[kt]*(a_all[kt]==1)*y_all[kt])/np.sum(sw_all[kt]*(a_all[kt]==1))
               - np.sum(sw_all[kt]*(a_all[kt]==0)*y_all[kt])/np.sum(sw_all[kt]*(a_all[kt]==0)))
msm = LogisticRegression(max_iter=1000)
msm.fit(a_all[kt].reshape(-1,1), y_all[kt], sample_weight=truncate(sw_all[kt]))
msm_rd = float(msm.predict_proba([[1]])[0,1] - msm.predict_proba([[0]])[0,1])
aipw_i = (mu1 - mu0) + a_all*(y_all-mu1)/e - (1-a_all)*(y_all-mu0)/(1-e)
ate_aipw = float(np.mean(aipw_i[kt]))

# --- bootstrap CIs for every estimator (reuses fixed nuisances; fast) ---
def _boot(fn, B=500, seed=CFG.RANDOM_SEED):
    rng = np.random.default_rng(seed); n = len(y_all); out = []
    for _ in range(B):
        idx = rng.integers(0, n, n); out.append(fn(idx))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

ci = {
 "g-formula (common support)":  _boot(lambda i: float(np.mean((mu1-mu0)[i][keep[i]]))),
 "stabilised IPW (common support)": _boot(lambda i: float(np.sum(sw_all[i][keep[i]]*(a_all[i][keep[i]]==1)*y_all[i][keep[i]])/max(np.sum(sw_all[i][keep[i]]*(a_all[i][keep[i]]==1)),1e-9)
                                          - np.sum(sw_all[i][keep[i]]*(a_all[i][keep[i]]==0)*y_all[i][keep[i]])/max(np.sum(sw_all[i][keep[i]]*(a_all[i][keep[i]]==0)),1e-9))),
 "AIPW (common support)":       _boot(lambda i: float(np.mean(aipw_i[i][keep[i]]))),
 "DR-ATO (overlap weights)":    _boot(lambda i: _ato(y_all, a_all, e, ow, mu1, mu0, i)[2]),
}

ate_table = pd.DataFrame([
 {"estimator":"g-formula (common support)","ATE_riskdiff":round(ate_g,4),"CI_lo":round(ci["g-formula (common support)"][0],4),"CI_hi":round(ci["g-formula (common support)"][1],4)},
 {"estimator":"stabilised IPW (common support)","ATE_riskdiff":round(ate_ipw,4),"CI_lo":round(ci["stabilised IPW (common support)"][0],4),"CI_hi":round(ci["stabilised IPW (common support)"][1],4)},
 {"estimator":"marginal structural model (common support)","ATE_riskdiff":round(msm_rd,4),"CI_lo":None,"CI_hi":None},
 {"estimator":"AIPW (common support)","ATE_riskdiff":round(ate_aipw,4),"CI_lo":round(ci["AIPW (common support)"][0],4),"CI_hi":round(ci["AIPW (common support)"][1],4)},
 {"estimator":"DR-ATO (overlap weights)","ATE_riskdiff":round(ate_ato_dr,4),"CI_lo":round(ci["DR-ATO (overlap weights)"][0],4),"CI_hi":round(ci["DR-ATO (overlap weights)"][1],4)},
])
ate_table.to_csv(CFG.TABLE_DIR / "ate_estimates.csv", index=False)

# --- FULL balance across all covariates + Love plot (overlap weights should ~zero every SMD) ---
smds_unw = np.array([smd(X_all[:, j], a_all)      for j in range(X_all.shape[1])])
smds_ow  = np.array([smd(X_all[:, j], a_all, ow)  for j in range(X_all.shape[1])])
smd_max_unw, smd_max_ow = float(np.nanmax(np.abs(smds_unw))), float(np.nanmax(np.abs(smds_ow)))
pd.DataFrame({"covariate":FEAT_NAMES,"SMD_unweighted":smds_unw,"SMD_overlap_weighted":smds_ow})\
  .to_csv(CFG.TABLE_DIR / "covariate_balance_full.csv", index=False)
figb, axb = plt.subplots(figsize=(6, 8))
order = np.argsort(np.abs(smds_unw))
axb.scatter(np.abs(smds_unw)[order], range(len(order)), s=8, color=PALETTE["neutral"], label="unweighted")
axb.scatter(np.abs(smds_ow)[order],  range(len(order)), s=8, color=PALETTE["harm"],   label="overlap-weighted")
axb.axvline(0.1, color="k", ls="--", lw=0.8); axb.set_xlabel("|SMD|"); axb.set_yticks([])
axb.set_title(f"Love plot: max |SMD| {smd_max_unw:.2f} -> {smd_max_ow:.2f}"); axb.legend(fontsize=8)
savefig(figb, "fig_love_balance")
logger.info("Common-support drop %.1f%% | max|SMD| %.2f -> %.2f (overlap)", 100*frac_dropped, smd_max_unw, smd_max_ow)
ate_table

00:08:45 | INFO    | loaded cross-fitted nuisance estimates from cache
00:09:01 | INFO    | figure -> fig_love_balance
00:09:01 | INFO    | Common-support drop 1.1% | max|SMD| 1.52 -> 0.00 (overlap)


,estimator,ATE_riskdiff,CI_lo,CI_hi
0,g-formula (common support),0.0745,0.0733,0.0758
1,stabilised IPW (common support),0.0603,0.0399,0.0790
2,marginal structural model (common support),0.0909,NaN,NaN
3,AIPW (common support),0.0641,0.0510,0.0754
4,DR-ATO (overlap weights),0.0125,-0.0101,0.0910


In [15]:
# =============================================================================
# E-value: sensitivity of the AIPW effect to unmeasured confounding
# Convert the marginal risk difference to an approximate risk ratio on mortality.
# =============================================================================
base_risk = float(y_all.mean())
r1 = np.clip(base_risk + ate_aipw * (1 - pt), 1e-4, 1 - 1e-4)   # crude marginal risks
r0 = np.clip(base_risk - ate_aipw * pt, 1e-4, 1 - 1e-4)
RR = r1 / r0
RR_star = RR if RR >= 1 else 1 / RR
def e_value(rr):
    return rr + np.sqrt(rr * (rr - 1))
ev = float(e_value(RR_star))
evalue = {"risk_ratio": float(RR), "RR_for_evalue": float(RR_star), "E_value": ev}
json.dump(evalue, open(CFG.TABLE_DIR / "evalue.json", "w"), indent=2)
logger.info("Approx marginal RR %.3f -> E-value %.2f", RR, ev)

# ATE forest + CATE distribution + E-value annotation
cate_all = mu1 - mu0
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
est_names = ate_table["estimator"].tolist(); est_vals = ate_table["ATE_riskdiff"].tolist()
yy = np.arange(len(est_names))
axes[0].scatter(est_vals, yy, color=PALETTE["accent"], zorder=3)
if "CI_lo" in ate_table:
    lo = ate_table["CI_lo"].dropna().iloc[0]; hi = ate_table["CI_hi"].dropna().iloc[0]
    axes[0].plot([lo, hi], [yy[-1], yy[-1]], color=PALETTE["accent"], lw=2)
axes[0].axvline(0, color="k", ls="--", lw=0.8)
axes[0].set_yticks(yy); axes[0].set_yticklabels(est_names, fontsize=9)
axes[0].set_xlabel("ATE on mortality (risk difference)"); axes[0].set_title("Average treatment effect")
axes[0].invert_yaxis()
axes[1].hist(cate_all, bins=40, color=PALETTE["neutral"], alpha=0.85)
axes[1].axvline(0, color="k", ls="--", lw=0.8)
axes[1].axvline(cate_all.mean(), color=PALETTE["harm"], lw=1.5, label=f"mean {cate_all.mean():.3f}")
axes[1].set_title(f"CATE distribution (E-value {ev:.2f})")
axes[1].set_xlabel("estimated treatment effect on mortality"); axes[1].legend()
savefig(fig, "fig_ate_cate")
ate_table


00:09:01 | INFO    | Approx marginal RR 1.781 -> E-value 2.96
00:09:02 | INFO    | figure -> fig_ate_cate


,estimator,ATE_riskdiff,CI_lo,CI_hi
0,g-formula (common support),0.0745,0.0733,0.0758
1,stabilised IPW (common support),0.0603,0.0399,0.0790
2,marginal structural model (common support),0.0909,NaN,NaN
3,AIPW (common support),0.0641,0.0510,0.0754
4,DR-ATO (overlap weights),0.0125,-0.0101,0.0910


In [16]:
# =============================================================================
# P3: Multi-learner CATE + external CATE validation
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier as HGBc
from sklearn.ensemble import HistGradientBoostingRegressor as HGBr
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

def c_for_benefit(pred_benefit, y, a, match_score, n_pairs=4000, seed=0):
    """van Klaveren c-for-benefit via 1:1 nearest-neighbour matching on a score."""
    rng = np.random.default_rng(seed)
    t_idx = np.where(a == 1)[0]; c_idx = np.where(a == 0)[0]
    if len(t_idx) == 0 or len(c_idx) == 0:
        return np.nan
    order = np.argsort(match_score[c_idx])
    cs_sorted = match_score[c_idx][order]; c_sorted = c_idx[order]
    pos = np.searchsorted(cs_sorted, match_score[t_idx])
    pos = np.clip(pos, 0, len(c_sorted) - 1)
    pairs_t = t_idx; pairs_c = c_sorted[pos]
    obs_ben = y[pairs_c] - y[pairs_t]
    prd_ben = 0.5 * (pred_benefit[pairs_t] + pred_benefit[pairs_c])
    i = rng.integers(0, len(pairs_t), n_pairs); k = rng.integers(0, len(pairs_t), n_pairs)
    ok = (prd_ben[i] != prd_ben[k]) & (obs_ben[i] != obs_ben[k])
    if ok.sum() == 0:
        return 0.5
    return float(((prd_ben[i] > prd_ben[k]) == (obs_ben[i] > obs_ben[k]))[ok].mean())

def fit_nuisance(Xtr, ytr, atr):
    e = HGBc().fit(Xtr, atr)
    m1 = HGBc().fit(Xtr[atr==1], ytr[atr==1])
    m0 = HGBc().fit(Xtr[atr==0], ytr[atr==0])
    return e, m1, m0

def cate_S(Xtr, ytr, atr):
    m = HGBc().fit(np.column_stack([Xtr, atr]), ytr)
    return lambda X: (m.predict_proba(np.column_stack([X, np.ones(len(X))]))[:,1]
                      - m.predict_proba(np.column_stack([X, np.zeros(len(X))]))[:,1])

def cate_T(Xtr, ytr, atr):
    m1 = HGBc().fit(Xtr[atr==1], ytr[atr==1])
    m0 = HGBc().fit(Xtr[atr==0], ytr[atr==0])
    return lambda X: m1.predict_proba(X)[:,1] - m0.predict_proba(X)[:,1]

def cate_X(Xtr, ytr, atr):
    e, m1, m0 = fit_nuisance(Xtr, ytr, atr)
    d1 = ytr[atr==1] - m0.predict_proba(Xtr[atr==1])[:,1]
    d0 = m1.predict_proba(Xtr[atr==0])[:,1] - ytr[atr==0]
    t1 = HGBr().fit(Xtr[atr==1], d1)
    t0 = HGBr().fit(Xtr[atr==0], d0)
    def f(X):
        ps = np.clip(e.predict_proba(X)[:,1], 1e-3, 1-1e-3)
        return ps*t0.predict(X) + (1-ps)*t1.predict(X)
    return f

def cate_DR(Xtr, ytr, atr):
    kf = StratifiedKFold(5, shuffle=True, random_state=CFG.RANDOM_SEED)
    mu1=np.zeros(len(ytr)); mu0=np.zeros(len(ytr)); e=np.zeros(len(ytr))
    for tr,te in kf.split(Xtr,ytr):
        e[te]=np.clip(HGBc().fit(Xtr[tr],atr[tr]).predict_proba(Xtr[te])[:,1],1e-3,1-1e-3)
        mu1[te]=HGBc().fit(Xtr[tr][atr[tr]==1],ytr[tr][atr[tr]==1]).predict_proba(Xtr[te])[:,1]
        mu0[te]=HGBc().fit(Xtr[tr][atr[tr]==0],ytr[tr][atr[tr]==0]).predict_proba(Xtr[te])[:,1]
    pseudo=(mu1-mu0)+atr*(ytr-mu1)/e-(1-atr)*(ytr-mu0)/(1-e)
    m=HGBr(max_iter=300).fit(Xtr,pseudo)
    return lambda X: m.predict(X)

# Fit all learners on development
learners = {"S": cate_S, "T": cate_T, "X": cate_X, "DR": cate_DR}
learner_fits = {}
for name, fit in learners.items():
    learner_fits[name] = fit(X_dev, y_dev, a_dev)
    logger.info("  trained %-14s", name)

# Generate benefits and validate
rows = []
for name, f in learner_fits.items():
    for coh, Xc, yc, ac, rc in [("MIMIC-hold", X_hold, y_hold, a_hold, r_hold),
                                ("eICU-ext",  X_ext,  y_ext,  a_ext,  r_ext)]:
        b = -f(Xc)
        cfb = c_for_benefit(b, yc, ac, rc, seed=CFG.RANDOM_SEED)
        rows.append({"learner": name, "cohort": coh,
                     "c_for_benefit": round(float(cfb), 3),
                     "benefit_sd": round(float(np.std(b)), 3),
                     "benefit_mean": round(float(np.mean(b)), 3)})
pd.DataFrame(rows).to_csv(CFG.TABLE_DIR / "cate_learner_validation.csv", index=False)
logger.info("CATE learner validation table written.")

# Use DR-learner as primary (most robust)
cate_of = learner_fits["DR"]
cate_hold = cate_of(X_hold); cate_ext = cate_of(X_ext)
benefit_hold = -cate_hold
benefit_ext = -cate_ext

logger.info("Audit CATE (DR-learner): hold mean %.3f | ext mean %.3f",
            cate_hold.mean(), cate_ext.mean())

np.savez(CFG.CACHE_DIR / "audit_inputs.npz",
         r_hold=r_hold, r_ext=r_ext, cate_hold=cate_hold, cate_ext=cate_ext,
         a_hold=a_hold, a_ext=a_ext, y_hold=y_hold, y_ext=y_ext,
         ps_hold=ps_hold, ps_ext=ps_ext, sw_hold=sw_hold, sw_ext=sw_ext)
print("Audit inputs cached.")

00:09:12 | INFO    |   trained S             
00:09:26 | INFO    |   trained T             
00:09:58 | INFO    |   trained X             
00:11:59 | INFO    |   trained DR            
00:12:04 | INFO    | CATE learner validation table written.
00:12:04 | INFO    | Audit CATE (DR-learner): hold mean 0.062 | ext mean 0.039
Audit inputs cached.


---
## 4. The Prescriptive Safety Audit

This is the contribution. We define a small family of metrics, each answering one distinct deployment question, all built on the same primitives from Sections 2 and 3: the treatment-blind risk $r_i = f(L_i)$, the estimated benefit $b_i = -\tau(L_i)$ (positive when treatment reduces mortality), the observed treatment $A_i$, and a risk-guided de-escalation rule at threshold $\tau_r$ that withholds support when $r_i < \tau_r$.

**Deployment change and its harm.** When the rule's action differs from observed care, deployment changes the patient's treatment. The change in expected mortality follows directly from the causal estimate:
$$\Delta_i(\tau_r) = \begin{cases} b_i & A_i = 1,\ r_i < \tau_r \quad \text{(withdraw support)} \\ -b_i & A_i = 0,\ r_i \ge \tau_r \quad \text{(add support)} \\ 0 & \text{no change.} \end{cases}$$
A change is **harmful** when $\Delta_i(\tau_r) > \delta$ for a clinically real margin $\delta$. Withdrawing support from a patient who benefits, or adding it where it raises mortality, both count.

**The metrics.**

1. **Treatment Influence Index**, $\mathrm{TII} = \mathbb{E}_L\,\lvert g(L, A{=}1) - g(L, A{=}0)\rvert$, where $g$ is a treatment-aware model. It measures how much predictive signal the treatment carries, that is, how much a model's output would move along the very axis deployment perturbs. It is the latent hazard, computed once, independent of threshold.

2. **Harmful Recommendation Rate**, $\mathrm{HRR}(\tau_r) = \frac{1}{N}\sum_i \mathbb{1}\{\Delta_i(\tau_r) > \delta\}$: the share of the whole cohort a deployment would harm.

3. **Harmful Flip Rate**, $\mathrm{HFR}(\tau_r) = \frac{\#\{\text{harmful changes}\}}{\#\{\text{changes}\}}$: among the patients whose care actually changes, the share harmed. This is the deployment-conditional hazard.

4. **Expected Excess Deaths**, $\mathrm{EED}(\tau_r) = \frac{1000}{N}\sum_i \Delta_i(\tau_r)$ per 1000 patients, reported both net and restricted to harmful changes, with bootstrap intervals. This is the tangible clinical quantity.

5. **Counterfactual Agreement Rate**, $\mathrm{CAR}(\tau_r) = \frac{1}{N}\sum_i \mathbb{1}\{(r_i \ge \tau_r) = (b_i > \delta)\}$, with a threshold-free companion: the rank concordance between risk and benefit. Low concordance is *benefit-risk inversion*, the signature of a prescriptively unsafe score, patients who are low risk precisely because treatment is keeping them alive.

6. **Deployment Hazard Index**, $\mathrm{DHI}$: the average positive net excess deaths per 1000 across the operating range, with $\mathrm{DHI}_{\text{frac}}$ the fraction of thresholds under net harm. It summarises magnitude, not just occurrence.

7. **Prescriptive Safety Score**, the headline $\mathrm{PSS} = 1 - \overline{\mathrm{HFR}}$, the mean safe share of deployment-induced changes over the operating range, bounded in $[0,1]$ and higher when safer.

We deliberately consolidated the wider set of candidate names (discordance indices, sensitivity scores, dependence scores) into these seven, because each remaining metric answers a question the others do not; redundant indices would invite the criticism of metric proliferation without added inferential content.

In [17]:
# =============================================================================
# Treatment Influence Index: how much signal the treatment itself carries
# =============================================================================
with timed("treatment-aware model + TII"):
    X_dev_aug  = np.column_stack([X_dev,  a_dev])
    X_hold_aug = np.column_stack([X_hold, a_hold])
    X_ext_aug  = np.column_stack([X_ext,  a_ext])
    g_model = CalibratedClassifierCV(boosting_clf(), method="isotonic", cv=3)
    g_model.fit(X_dev_aug, y_dev)

def tii(Xaug):
    X1 = Xaug.copy(); X1[:, -1] = 1
    X0 = Xaug.copy(); X0[:, -1] = 0
    g1 = g_model.predict_proba(X1)[:, 1]; g0 = g_model.predict_proba(X0)[:, 1]
    return g1 - g0

tii_hold = tii(X_hold_aug); tii_ext = tii(X_ext_aug)
# treatment-aware vs treatment-blind discrimination gain
auc_blind = roc_auc_score(y_hold, r_hold)
auc_aware = roc_auc_score(y_hold, g_model.predict_proba(X_hold_aug)[:, 1])
tii_summary = {"TII_hold_mean_abs": float(np.mean(np.abs(tii_hold))),
               "TII_hold_signed_mean": float(np.mean(tii_hold)),
               "TII_ext_mean_abs": float(np.mean(np.abs(tii_ext))),
               "AUROC_blind": float(auc_blind), "AUROC_treatment_aware": float(auc_aware),
               "delta_AUROC": float(auc_aware - auc_blind)}
json.dump(tii_summary, open(CFG.TABLE_DIR / "tii_summary.json", "w"), indent=2)
logger.info("TII (hold) mean|.| = %.4f | treatment-aware AUROC gain = %.4f",
            tii_summary["TII_hold_mean_abs"], tii_summary["delta_AUROC"])
tii_summary


00:12:04 | INFO    | START  treatment-aware model + TII
00:12:10 | INFO    | DONE   treatment-aware model + TII  (6.4s)
00:12:13 | INFO    | TII (hold) mean|.| = 0.0508 | treatment-aware AUROC gain = 0.0034


{'TII_hold_mean_abs': 0.05076336854466624,
 'TII_hold_signed_mean': 0.04902557619734836,
 'TII_ext_mean_abs': 0.040651926538155476,
 'AUROC_blind': 0.9088180001736216,
 'AUROC_treatment_aware': 0.9121789034530182,
 'delta_AUROC': 0.003360903279396643}

In [18]:
# =============================================================================
# Framework core: per-threshold metrics, scalar summaries, bootstrap intervals
# =============================================================================
DELTA = CFG.BENEFIT_DELTA
THR = CFG.THRESHOLDS

def deployment_delta(r, benefit, a, tau):
    action = (r >= tau).astype(int)
    withdraw = (a == 1) & (action == 0)
    add      = (a == 0) & (action == 1)
    delta = np.where(withdraw, benefit, np.where(add, -benefit, 0.0))
    flip = action != a
    return delta, flip

def metrics_at(r, benefit, a, tau):
    delta, flip = deployment_delta(r, benefit, a, tau)
    harmful = flip & (delta > DELTA)
    n = len(r)
    hrr = float(harmful.mean())
    hfr = float(harmful.sum() / flip.sum()) if flip.sum() > 0 else np.nan
    eed_net = float(delta.sum() / n * 1000)
    eed_harm = float(delta[harmful].sum() / n * 1000)
    car = float(np.mean((r >= tau) == (benefit > DELTA)))
    return hrr, hfr, eed_net, eed_harm, car, harmful

def sweep(r, benefit, a, thresholds=THR):
    rows = [metrics_at(r, benefit, a, t) for t in thresholds]
    keys = ["hrr", "hfr", "eed_net", "eed_harm", "car"]
    out = {k: np.array([row[i] for row in rows]) for i, k in enumerate(keys)}
    harmful_sets = [row[5] for row in rows]           # NB2's metrics_at returns 6 (incl. harmful)
    n = len(r); eed_wd, eed_add = [], []
    for t in thresholds:
        delta, _ = deployment_delta(r, benefit, a, t)
        action = (r >= t).astype(int)
        withdraw = (a == 1) & (action == 0); add = (a == 0) & (action == 1)
        eed_wd.append(float(delta[withdraw].sum() / n * 1000))
        eed_add.append(float(delta[add].sum() / n * 1000))
    out["eed_wd"] = np.array(eed_wd); out["eed_add"] = np.array(eed_add)
    return out, harmful_sets            # <-- TUPLE, unlike NB3

def scalars(sw):
    eed = sw["eed_net"]
    return {"PSS": float(np.nanmean(1 - sw["hfr"])),
            "DHI": float(np.mean(eed)),                          # canonical: SIGNED net EED/1000
            "DHI_onesided": float(np.mean(np.maximum(eed, 0))),  # reference only, NOT headline
            "DHI_signed": float(np.mean(eed)),
            "DHI_frac_positive": float(np.mean(eed > 0)),
            "DHI_withdraw": float(np.mean(sw["eed_wd"])),        # signed, not max(,0)
            "DHI_add": float(np.mean(sw["eed_add"])),           # signed, not max(,0)
            "DHI_frac": float(np.mean(eed > 0)),
            "CAR_mean": float(np.nanmean(sw["car"]))}

def rank_concordance(r, benefit, n_pairs=20000, seed=CFG.RANDOM_SEED):
    rng = np.random.default_rng(seed); n = len(r)
    i = rng.integers(0, n, n_pairs); j = rng.integers(0, n, n_pairs)
    ok = (r[i] != r[j]) & (benefit[i] != benefit[j])
    conc = ((r[i] > r[j]) == (benefit[i] > benefit[j]))[ok].mean()
    return float(conc)

# compute for every model on both cohorts (benefit is model-agnostic, from the causal engine)
audit_rows = []
sweeps = {}
for coh, rdict, benefit, a in [("MIMIC-hold", risk_hold, benefit_hold, a_hold),
                               ("eICU-ext",  risk_ext,  benefit_ext,  a_ext)]:
    for name, r in rdict.items():
        sw, hsets = sweep(r, benefit, a)
        sc = scalars(sw); sc["concordance_r_b"] = rank_concordance(r, benefit)
        sc.update({"model": name, "cohort": coh})
        audit_rows.append(sc); sweeps[(coh, name)] = (sw, hsets)

audit = pd.DataFrame(audit_rows)[["model", "cohort", "PSS", "DHI", "DHI_frac",
                                  "CAR_mean", "concordance_r_b"]].round(4)
audit.to_csv(CFG.TABLE_DIR / "prescriptive_safety_scores.csv", index=False)

# bootstrap CIs for the primary model headline on each cohort
def boot_scalar(r, benefit, a, key, B=min(CFG.N_BOOTSTRAP, 150), seed=CFG.RANDOM_SEED):
    rng = np.random.default_rng(seed); n = len(r); vals = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        sw, _ = sweep(r[idx], benefit[idx], a[idx])
        vals.append(scalars(sw)[key])
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

head_ci = []
for coh, r, benefit, a in [("MIMIC-hold", r_hold, benefit_hold, a_hold),
                           ("eICU-ext",  r_ext,  benefit_ext,  a_ext)]:
    for key in ["PSS", "DHI"]:
        m, lo, hi = boot_scalar(r, benefit, a, key)
        head_ci.append({"cohort": coh, "metric": key, "mean": round(m, 4),
                        "CI_lo": round(lo, 4), "CI_hi": round(hi, 4)})
head_ci = pd.DataFrame(head_ci)
head_ci.to_csv(CFG.TABLE_DIR / "headline_ci.csv", index=False)
logger.info("Prescriptive safety scores computed for %d model-cohort pairs.", len(audit))
audit


# =============================================================================
# P7: FDR control over the threshold sweep
# =============================================================================
from statsmodels.stats.multitest import multipletests

def sweep_fdr(r, benefit, a, thresholds=THR, B=300, seed=CFG.RANDOM_SEED):
    """Sweep with FDR-corrected p-values."""
    rng = np.random.default_rng(seed)
    n = len(r)
    
    # Observed EED per threshold
    obs_eed = np.array([deployment_delta(r, benefit, a, t)[0].sum() / n * 1000 for t in thresholds])
    
    # Bootstrap null: EED under random permutation of benefit
    boot_eed = np.zeros((B, len(thresholds)))
    for b in range(B):
        b_perm = rng.permutation(benefit)
        boot_eed[b, :] = np.array([deployment_delta(r, b_perm, a, t)[0].sum() / n * 1000 for t in thresholds])
    
    # Two-sided p-values
    p = 2 * np.minimum(
        np.mean(boot_eed >= obs_eed, axis=0),
        np.mean(boot_eed <= obs_eed, axis=0)
    )
    p = np.clip(p, 1/(B+1), 1.0)
    
    # BH correction
    rej, p_bh, _, _ = multipletests(p, method="fdr_bh")
    
    return thresholds, obs_eed, p, p_bh, rej

# Run FDR sweep on hold-out
thr, eed, p_raw, p_bh, rej = sweep_fdr(r_hold, benefit_hold, a_hold)

fdr_df = pd.DataFrame({
    "threshold": thr,
    "EED_net": eed,
    "p_raw": p_raw,
    "p_bh": p_bh,
    "reject_BH": rej
})
fdr_df.to_csv(CFG.TABLE_DIR / "fdr_sweep.csv", index=False)
logger.info("FDR sweep complete. %d thresholds reject BH at alpha=0.05", rej.sum())

00:13:31 | INFO    | Prescriptive safety scores computed for 10 model-cohort pairs.
00:13:32 | INFO    | FDR sweep complete. 33 thresholds reject BH at alpha=0.05


In [19]:
# =============================================================================
# P5: Single-modality treatment audits (CORRECTED)
# =============================================================================
def run_treatment_audit(TREAT, Xd, yd, ad, Xh, yh, ah, Xe, ye, ae, rh, re):
    """
    Run the audit for a single treatment definition.
    rh: risk scores for MIMIC hold-out (internal validation)
    re: risk scores for eICU (external validation)
    """
    from sklearn.ensemble import HistGradientBoostingRegressor
    
    # Ensure all inputs are numpy arrays
    rh = np.asarray(rh).ravel()
    re = np.asarray(re).ravel()
    ah = np.asarray(ah).ravel()
    yh = np.asarray(yh).ravel()
    ae = np.asarray(ae).ravel()
    ye = np.asarray(ye).ravel()
    
    # Check lengths
    if len(rh) != len(ah) or len(rh) != len(yh):
        logger.warning("Hold-out length mismatch for %s: rh=%d, ah=%d, yh=%d. Skipping internal.",
                       TREAT, len(rh), len(ah), len(yh))
        return None
    
    if len(re) != len(ae) or len(re) != len(ye):
        logger.warning("External length mismatch for %s: re=%d, ae=%d, ye=%d. Skipping external.",
                       TREAT, len(re), len(ae), len(ye))
        return None
    
    # Refit DR-learner for this treatment
    kf = StratifiedKFold(5, shuffle=True, random_state=CFG.RANDOM_SEED)
    mu1 = np.zeros(len(yd)); mu0 = np.zeros(len(yd)); e = np.zeros(len(yd))
    for tr, te in kf.split(Xd, yd):
        e[te] = np.clip(HGBc().fit(Xd[tr], ad[tr]).predict_proba(Xd[te])[:,1], 1e-3, 1-1e-3)
        t1 = tr[ad[tr]==1]; t0 = tr[ad[tr]==0]
        if len(t1) > 10 and len(t0) > 10:
            mu1[te] = HGBc().fit(Xd[t1], yd[t1]).predict_proba(Xd[te])[:,1]
            mu0[te] = HGBc().fit(Xd[t0], yd[t0]).predict_proba(Xd[te])[:,1]
        else:
            mu1[te] = yd.mean(); mu0[te] = yd.mean()
    pseudo = (mu1-mu0) + ad*(yd-mu1)/e - (1-ad)*(yd-mu0)/(1-e)
    dr = HistGradientBoostingRegressor(max_iter=300).fit(Xd, pseudo)
    
    # Benefits
    b_hold = -dr.predict(Xh)
    b_ext = -dr.predict(Xe)
    
    # --- Internal audit (MIMIC hold-out) ---
    try:
        cfb_hold = c_for_benefit(b_hold, yh, ah, rh, seed=CFG.RANDOM_SEED)
    except Exception as e:
        logger.warning("c_for_benefit hold failed for %s: %s", TREAT, e)
        cfb_hold = np.nan
    
    try:
        sw_h, _ = sweep(rh, b_hold, ah)
        sc_h = scalars(sw_h)
    except Exception as e:
        logger.warning("Sweep hold failed for %s: %s", TREAT, e)
        sc_h = {"PSS": np.nan, "DHI_signed": np.nan}
    
    # --- External audit (eICU) ---
    try:
        cfb_ext = c_for_benefit(b_ext, ye, ae, re, seed=CFG.RANDOM_SEED)
    except Exception as e:
        logger.warning("c_for_benefit ext failed for %s: %s", TREAT, e)
        cfb_ext = np.nan
    
    try:
        sw_e, _ = sweep(re, b_ext, ae)
        sc_e = scalars(sw_e)
    except Exception as e:
        logger.warning("Sweep ext failed for %s: %s", TREAT, e)
        sc_e = {"PSS": np.nan, "DHI_signed": np.nan}
    
    return {
        "treatment": TREAT,
        "n_treated_hold": int(ah.sum()),
        "n_treated_ext": int(ae.sum()),
        "c_for_benefit_hold": round(float(cfb_hold), 3) if not np.isnan(cfb_hold) else np.nan,
        "c_for_benefit_ext": round(float(cfb_ext), 3) if not np.isnan(cfb_ext) else np.nan,
        "PSS_hold": round(sc_h["PSS"], 4) if not np.isnan(sc_h["PSS"]) else np.nan,
        "PSS_ext": round(sc_e["PSS"], 4) if not np.isnan(sc_e["PSS"]) else np.nan,
        "DHI_hold": round(sc_h["DHI_signed"], 4) if not np.isnan(sc_h["DHI_signed"]) else np.nan,
        "DHI_ext": round(sc_e["DHI_signed"], 4) if not np.isnan(sc_e["DHI_signed"]) else np.nan,
    }

# Run for each treatment
treatments = ["vaso_post", "vent_post", "rrt_post", "any_organ_support_post"]
modality_results = []

for t in treatments:
    logger.info("Running audit for treatment: %s", t)
    # Extract treatment indicators
    if t == "any_organ_support_post":
        a_dev_t = a_dev
        a_hold_t = a_hold
        a_ext_t = a_ext
    else:
        a_dev_t = mimic_dev[t].values.astype(int) if t in mimic_dev.columns else a_dev
        a_hold_t = mimic_hold[t].values.astype(int) if t in mimic_hold.columns else a_hold
        a_ext_t = eicu[t].values.astype(int) if t in eicu.columns else a_ext
    
    # Skip if lengths don't match (shouldn't happen for correctly aligned data)
    if len(a_hold_t) != len(r_hold):
        logger.warning("Skipping %s: hold length %d != r_hold %d", t, len(a_hold_t), len(r_hold))
        continue
    if len(a_ext_t) != len(r_ext):
        logger.warning("Skipping %s: ext length %d != r_ext %d", t, len(a_ext_t), len(r_ext))
        continue
    
    try:
        res = run_treatment_audit(t, X_dev, y_dev, a_dev_t, X_hold, y_hold, a_hold_t, 
                                  X_ext, y_ext, a_ext_t, r_hold, r_ext)
        if res is not None:
            modality_results.append(res)
    except Exception as e:
        logger.warning("Treatment %s failed: %s", t, e)

# Save results
if modality_results:
    pd.DataFrame(modality_results).to_csv(CFG.TABLE_DIR / "modality_audits.csv", index=False)
    logger.info("Modality audits complete. %d results.", len(modality_results))
else:
    logger.warning("No modality audit results. Check data alignment.")

00:13:32 | INFO    | Running audit for treatment: vaso_post
00:15:33 | INFO    | Running audit for treatment: vent_post
00:17:46 | INFO    | Running audit for treatment: rrt_post
00:19:31 | INFO    | Running audit for treatment: any_organ_support_post
00:21:24 | INFO    | Modality audits complete. 4 results.


In [20]:
# =============================================================================
# P9: Alternative decision rules + clinician non-compliance
# =============================================================================
def sweep_alternative(r, benefit, a, thresholds=THR, rule="de-escalation", compliance=1.0, seed=CFG.RANDOM_SEED):
    """Sweep with alternative rules and clinician non-compliance."""
    rng = np.random.default_rng(seed)
    n = len(r)
    results = []
    
    for tau in thresholds:
        # Base action
        base_action = (r >= tau).astype(int)
        
        # Apply compliance: clinicians follow the rule with probability 'compliance'
        if compliance < 1.0:
            follow = rng.random(n) < compliance
            action = np.where(follow, base_action, a)
        else:
            action = base_action
        
        # Alternative rules
        if rule == "de-escalation":
            # Withhold support for low risk (original)
            withdraw = (a == 1) & (action == 0)
            add = (a == 0) & (action == 1)
        elif rule == "treat-high-risk":
            # Treat low-risk patients (opposite of de-escalation)
            # De-escalation: treat if r >= tau (withhold if r < tau)
            # Treat-low-risk: treat if r < tau
            action = (r < tau).astype(int)  
            withdraw = (a == 1) & (action == 0)
            add = (a == 0) & (action == 1)
        elif rule == "benefit-guided":
            # Treat if benefit > delta
            benefit_action = (benefit > DELTA).astype(int)
            withdraw = (a == 1) & (benefit_action == 0)
            add = (a == 0) & (benefit_action == 1)
            action = benefit_action
        
        delta = np.where(withdraw, benefit, np.where(add, -benefit, 0.0))
        flip = action != a
        harmful = flip & (delta > DELTA)
        
        results.append({
            "tau": tau,
            "eed_net": float(delta.sum() / n * 1000),
            "hrr": float(harmful.mean()),
            "hfr": float(harmful.sum() / flip.sum()) if flip.sum() > 0 else np.nan,
            "flip_rate": float(flip.mean()),
            "n_flips": int(flip.sum()),
            "n_harmful": int(harmful.sum())
        })
    
    return pd.DataFrame(results)

# Run for different compliance levels
compliance_levels = [1.0, 0.75, 0.5, 0.25]
alt_results = []
for comp in compliance_levels:
    df = sweep_alternative(r_hold, benefit_hold, a_hold, rule="de-escalation", compliance=comp)
    df["compliance"] = comp
    alt_results.append(df)
    logger.info("Compliance %.2f: DHI=%.3f, HRR=%.3f", 
                comp, 
                scalars({"eed_net": df["eed_net"].values, "hfr": df["hfr"].values, 
                         "eed_wd": np.zeros_like(df["eed_net"]), "eed_add": np.zeros_like(df["eed_net"]),
                         "car": np.ones_like(df["eed_net"])})["DHI"],
                df["hrr"].mean())

alt_df = pd.concat(alt_results, ignore_index=True)
alt_df.to_csv(CFG.TABLE_DIR / "alternative_rules_compliance.csv", index=False)
logger.info("Alternative rules and compliance analysis complete.")

00:21:24 | INFO    | Compliance 1.00: DHI=-10.161, HRR=0.055
00:21:24 | INFO    | Compliance 0.75: DHI=-7.615, HRR=0.041
00:21:24 | INFO    | Compliance 0.50: DHI=-5.018, HRR=0.027
00:21:24 | INFO    | Compliance 0.25: DHI=-2.495, HRR=0.014
00:21:24 | INFO    | Alternative rules and compliance analysis complete.


In [21]:
# =============================================================================
# Figure: Deployment Safety Frontier, Prescriptive Reliability, Decision Stability
# =============================================================================
sw_h, hsets_h = sweeps[("MIMIC-hold", PRIMARY)]
sw_e, hsets_e = sweeps[("eICU-ext", PRIMARY)]

def jaccard_stability(hsets):
    out = []
    for k in range(1, len(hsets)):
        a_, b_ = hsets[k-1], hsets[k]
        u = (a_ | b_).sum()
        out.append((a_ & b_).sum() / u if u > 0 else 1.0)
    return np.array(out)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
# 1) Deployment Safety Frontier
for sw, lab, c in [(sw_h, "MIMIC hold", PALETTE["MIMIC"]), (sw_e, "eICU ext", PALETTE["eICU"])]:
    axes[0].plot(THR, sw["eed_net"], lw=1.8, color=c, label=lab)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].fill_between(THR, 0, np.maximum(sw_h["eed_net"], 0), color=PALETTE["harm"], alpha=0.15)
# frontier crossing for MIMIC
cross = THR[np.argmax(sw_h["eed_net"] <= 0)] if np.any(sw_h["eed_net"] <= 0) else THR[-1]
axes[0].axvline(cross, color=PALETTE["harm"], ls=":", lw=1.2, label=f"frontier tau={cross:.2f}")
axes[0].set_title("Deployment Safety Frontier")
axes[0].set_xlabel("risk threshold tau"); axes[0].set_ylabel("net excess deaths per 1000"); axes[0].legend(fontsize=8)
# 2) Prescriptive Reliability Curve
axes[1].plot(THR, 100*sw_h["hrr"], color=PALETTE["harm"], lw=1.8, label="HRR (% cohort harmed)")
axes[1].plot(THR, sw_h["eed_harm"], color=PALETTE["accent"], lw=1.8, ls="--", label="harmful EED /1000")
axes[1].set_title("Prescriptive Reliability Curve (MIMIC hold)")
axes[1].set_xlabel("risk threshold tau"); axes[1].set_ylabel("value"); axes[1].legend(fontsize=8)
# 3) Decision Stability Curve
axes[2].plot(THR, [h.sum() for h in hsets_h], color=PALETTE["neutral"], lw=1.8, label="harmful count")
ax2b = axes[2].twinx()
ax2b.plot(THR[1:], jaccard_stability(hsets_h), color=PALETTE["MIMIC"], lw=1.4, ls=":", label="Jaccard stability")
axes[2].set_title("Decision Stability Curve"); axes[2].set_xlabel("risk threshold tau")
axes[2].set_ylabel("harmful patients (count)"); ax2b.set_ylabel("adjacent-threshold Jaccard")
lines = axes[2].get_lines() + ax2b.get_lines(); axes[2].legend(lines, [l.get_label() for l in lines], fontsize=8)
savefig(fig, "fig_deployment_safety_curves")
print("Deployment safety curves written.")


00:21:26 | INFO    | figure -> fig_deployment_safety_curves
Deployment safety curves written.


In [22]:
# =============================================================================
# Figure: benefit-risk inversion (CAR) and the recommendation disagreement matrix
# =============================================================================
tau_ref = float(y_hold.mean())   # a clinically natural operating point: outcome prevalence
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))

# scatter: predicted risk vs estimated benefit, coloured by observed treatment
sc = axes[0].scatter(r_hold, benefit_hold, c=a_hold, cmap="coolwarm", s=12, alpha=0.6)
axes[0].axvline(tau_ref, color="k", ls="--", lw=0.9)
axes[0].axhline(DELTA, color="grey", ls=":", lw=0.9)
# highlight harmful de-escalation region: low risk, high benefit, treated
harm_mask = (r_hold < tau_ref) & (benefit_hold > DELTA) & (a_hold == 1)
axes[0].scatter(r_hold[harm_mask], benefit_hold[harm_mask], facecolors="none",
                edgecolors=PALETTE["harm"], s=40, lw=1.2, label=f"harmful de-escalation (n={harm_mask.sum()})")
axes[0].set_xlabel("predicted risk r"); axes[0].set_ylabel("estimated benefit b = -CATE")
axes[0].set_title(f"Benefit-risk inversion (tau={tau_ref:.2f})")
axes[0].legend(fontsize=8); plt.colorbar(sc, ax=axes[0], label="observed treatment A")

# 2x2 recommendation disagreement matrix at tau_ref
action = (r_hold >= tau_ref).astype(int)
M = np.zeros((2, 2))
for ai in (0, 1):
    for pi in (0, 1):
        M[ai, pi] = np.sum((a_hold == ai) & (action == pi))
im = axes[1].imshow(M, cmap="Purples")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["policy withhold", "policy treat"])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["observed untreated", "observed treated"])
for ai in (0, 1):
    for pi in (0, 1):
        harmed = ""
        if ai == 1 and pi == 0:
            hh = int(((a_hold==1)&(action==0)&(benefit_hold>DELTA)).sum()); harmed = f"\n({hh} harmful)"
        if ai == 0 and pi == 1:
            hh = int(((a_hold==0)&(action==1)&(-benefit_hold>DELTA)).sum()); harmed = f"\n({hh} harmful)"
        axes[1].text(pi, ai, f"{int(M[ai, pi])}{harmed}", ha="center", va="center",
                     color="black", fontsize=10)
axes[1].set_title("Recommendation disagreement matrix")
plt.colorbar(im, ax=axes[1], fraction=0.046)
savefig(fig, "fig_inversion_disagreement")
print(f"Inversion + disagreement figure written (harmful de-escalations at tau_ref: {harm_mask.sum()}).")


00:21:30 | INFO    | figure -> fig_inversion_disagreement
Inversion + disagreement figure written (harmful de-escalations at tau_ref: 46).


In [23]:
# =============================================================================
# Figure: excess-deaths waterfall and treated-to-action Sankey
# =============================================================================
# waterfall: net excess deaths decomposed by change type at tau_ref
action = (r_hold >= tau_ref).astype(int)
withdraw = (a_hold == 1) & (action == 0)
add      = (a_hold == 0) & (action == 1)
comp = {
    "harmful withdrawals": benefit_hold[withdraw & (benefit_hold > DELTA)].sum(),
    "beneficial withdrawals": benefit_hold[withdraw & (benefit_hold <= DELTA)].sum(),
    "harmful additions": (-benefit_hold[add & (-benefit_hold > DELTA)]).sum(),
    "beneficial additions": (-benefit_hold[add & (-benefit_hold <= DELTA)]).sum(),
}
comp = {k: float(v) / len(r_hold) * 1000 for k, v in comp.items()}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = list(comp.keys()); vals = list(comp.values())
cumulative = np.cumsum([0] + vals[:-1])
colors = [PALETTE["harm"], PALETTE["safe"], PALETTE["harm"], PALETTE["safe"]]
for i, (lab, v, base) in enumerate(zip(labels, vals, cumulative)):
    axes[0].bar(i, v, bottom=base, color=colors[i], alpha=0.85)
axes[0].axhline(sum(vals), color="k", ls="--", lw=1, label=f"net {sum(vals):.1f}/1000")
axes[0].set_xticks(range(len(labels))); axes[0].set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("excess deaths per 1000"); axes[0].set_title(f"Excess-deaths waterfall (tau={tau_ref:.2f})")
axes[0].legend(fontsize=8)

# Sankey: observed treated flow into policy actions and harm
if _HAS_SANKEY:
    n_treated = int((a_hold == 1).sum())
    n_wd = int(withdraw.sum()); n_keep = n_treated - n_wd
    n_wd_harm = int((withdraw & (benefit_hold > DELTA)).sum()); n_wd_ok = n_wd - n_wd_harm
    try:
        sk = Sankey(ax=axes[1], unit=None, scale=1.0/max(n_treated,1), gap=0.4)
        sk.add(flows=[n_treated, -n_keep, -n_wd],
               labels=["observed treated", "care unchanged", "support withdrawn"],
               orientations=[0, 0, 1], trunklength=6)
        sk.finish()
        axes[1].set_title(f"Treated -> policy action (withdrawn: {n_wd}, of which harmful: {n_wd_harm})")
    except Exception as ex:
        axes[1].text(0.5, 0.5, f"Sankey unavailable: {ex}", ha="center"); axes[1].axis("off")
else:
    axes[1].text(0.5, 0.5, "matplotlib Sankey unavailable", ha="center"); axes[1].axis("off")
axes[1].axis("off")
savefig(fig, "fig_waterfall_sankey")
print("Waterfall + Sankey written.")


00:21:31 | INFO    | figure -> fig_waterfall_sankey
Waterfall + Sankey written.


---
## 5. Interpretability and the deployment feedback loop

A prescriptively unsafe model is not a black box misfiring at random; it is keying on features that also drive treatment. We make that visible. We compute global attributions for the treatment-blind risk model, attributions for the treatment-aware model (where the treatment's own rank is revealing), and the overlap between the features that drive predicted risk and the features that drive treatment assignment. When the two attribution profiles coincide, the risk score is effectively a treatment surrogate, which is the mechanism by which deploying it perturbs care. Partial-dependence and individual conditional expectation curves show how predicted risk responds to the treatment axis directly.

In [24]:
# =============================================================================
# Global attributions: SHAP where available, permutation importance otherwise
# =============================================================================
from sklearn.inspection import permutation_importance, partial_dependence

rng = np.random.default_rng(CFG.RANDOM_SEED)
samp = rng.choice(len(X_hold), size=min(CFG.SHAP_SAMPLE, len(X_hold)), replace=False)

def global_importance(estimator, X_tr, y_tr, X_ev, y_ev, names, sample_idx, shap_base=None):
    # SHAP on an uncalibrated learner refit on the SAME design as X_ev; permutation otherwise.
    if _HAS_SHAP and shap_base is not None:
        try:
            base_fit = clone(shap_base).fit(X_tr, y_tr)
            bg = X_tr[np.random.default_rng(CFG.RANDOM_SEED).choice(
                len(X_tr), min(200, len(X_tr)), replace=False)]
            sv = shap.Explainer(base_fit, bg)(X_ev[sample_idx], check_additivity=False)
            vals = np.abs(sv.values)
            while vals.ndim > 1:                       # collapse any class/output axes
                vals = vals.mean(-1) if vals.shape[-1] != len(names) else vals.mean(0)
                if vals.ndim == 1:
                    break
            return pd.Series(np.ravel(vals), index=names).sort_values(ascending=False), sv
        except Exception as ex:
            logger.warning("SHAP failed (%s); using permutation importance", ex)
    r = permutation_importance(estimator, X_ev[sample_idx], y_ev[sample_idx],
                               n_repeats=5, random_state=CFG.RANDOM_SEED, n_jobs=CFG.N_JOBS)
    return pd.Series(r.importances_mean, index=names).sort_values(ascending=False), None

imp_risk, sv_risk = global_importance(models[PRIMARY], X_dev, y_dev, X_hold, y_hold,
                                      FEAT_NAMES, samp, shap_base=boosting_clf())
imp_risk.head(20).to_csv(CFG.TABLE_DIR / "importance_risk_model.csv")

# propensity attributions to expose treatment surrogates
imp_ps, _ = global_importance(ps_model, X_dev, a_dev, X_hold, a_hold,
                              FEAT_NAMES, samp, shap_base=boosting_clf())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
imp_risk.head(15).iloc[::-1].plot(kind="barh", ax=axes[0], color=PALETTE["MIMIC"])
axes[0].set_title(f"Top risk drivers ({'SHAP' if _HAS_SHAP else 'permutation'})"); axes[0].set_xlabel("importance")
imp_ps.head(15).iloc[::-1].plot(kind="barh", ax=axes[1], color=PALETTE["harm"])
axes[1].set_title("Top treatment (propensity) drivers"); axes[1].set_xlabel("importance")
savefig(fig, "fig_attributions")

# overlap between risk drivers and treatment drivers = surrogate signal
k = 15
top_risk = set(imp_risk.head(k).index); top_ps = set(imp_ps.head(k).index)
overlap = top_risk & top_ps
feedback = {"top_k": k, "overlap_count": len(overlap),
            "overlap_fraction": len(overlap) / k, "overlap_features": sorted(overlap)}
json.dump(feedback, open(CFG.TABLE_DIR / "risk_treatment_overlap.json", "w"), indent=2)
logger.info("Risk/treatment driver overlap: %d of %d (%.0f%%)",
            len(overlap), k, 100 * len(overlap) / k)
feedback


00:23:00 | INFO    | figure -> fig_attributions
00:23:00 | INFO    | Risk/treatment driver overlap: 0 of 15 (0%)


{'top_k': 15,
 'overlap_count': 0,
 'overlap_fraction': 0.0,
 'overlap_features': []}

In [25]:
# =============================================================================
# Treatment-aware SHAP rank of A, and PDP/ICE along the treatment axis
# =============================================================================
aug_names = FEAT_NAMES + ["TREATMENT_A"]
imp_aware, _ = global_importance(g_model, X_dev_aug, y_dev, X_hold_aug, y_hold,
                                 aug_names, samp, shap_base=boosting_clf())
rank_A = list(imp_aware.index).index("TREATMENT_A") + 1 if "TREATMENT_A" in imp_aware.index else -1
logger.info("Treatment variable ranks %d of %d in the treatment-aware model", rank_A, len(aug_names))

# PDP + ICE for the top risk feature, and PDP for the treatment axis in g
top_feat = imp_risk.index[0]; jf = FEAT_NAMES.index(top_feat)
base_fit = clone(models[PRIMARY].estimator if hasattr(models[PRIMARY], "estimator") else boosting_clf()).fit(X_dev, y_dev)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
try:
    pd_res = partial_dependence(base_fit, X_hold, [jf], kind="both", grid_resolution=25)
    gx = pd_res["grid_values"][0]
    ice = pd_res["individual"][0]
    for c in rng.choice(ice.shape[0], size=min(60, ice.shape[0]), replace=False):
        axes[0].plot(gx, ice[c], color="grey", alpha=0.25, lw=0.6)
    axes[0].plot(gx, pd_res["average"][0], color=PALETTE["MIMIC"], lw=2.2, label="PDP")
    axes[0].set_title(f"PDP / ICE: {top_feat}"); axes[0].set_xlabel(top_feat)
    axes[0].set_ylabel("predicted risk"); axes[0].legend(fontsize=8)
except Exception as ex:
    axes[0].text(0.5, 0.5, f"PDP unavailable: {ex}", ha="center"); axes[0].axis("off")

# effect of toggling the treatment axis in the treatment-aware model
axes[1].hist(tii_hold, bins=40, color=PALETTE["accent"], alpha=0.85)
axes[1].axvline(0, color="k", ls="--", lw=0.8)
axes[1].set_title("Change in predicted risk when A is toggled (g)")
axes[1].set_xlabel("g(L, A=1) - g(L, A=0)"); axes[1].set_ylabel("patients")
savefig(fig, "fig_pdp_ice_treatment")
print("PDP/ICE and treatment-toggle figure written.")


00:23:41 | INFO    | Treatment variable ranks 1 of 325 in the treatment-aware model
00:23:45 | INFO    | figure -> fig_pdp_ice_treatment
PDP/ICE and treatment-toggle figure written.


---
## 6. Stress testing and robustness

A methodology that only holds in one configuration is not deployable. We subject the audit to the perturbations a reviewer will demand: external transportability (already built into every table above), subgroup fairness of prescriptive harm, sensitivity to the risk threshold (the full curves in Section 4), sensitivity to treatment prevalence, robustness to additional missingness, an ablation that removes treatment-surrogate features to test whether the hazard is mitigable, and the effect of calibration drift on prescriptive safety. Each returns the same headline quantities, so the reader can see directly how far they move.

In [26]:
# =============================================================================
# Subgroup fairness of prescriptive harm (sex, race, age bands)
# =============================================================================
hold_meta = mimic_hold.reset_index(drop=True).copy()
hold_meta["age_band"] = pd.cut(hold_meta["admission_age"], [0, 50, 65, 80, 200],
                               labels=["<50", "50-64", "65-79", "80+"])
sub_rows = []
def subgroup_scalar(mask):
    if mask.sum() < 40:
        return None
    sw, _ = sweep(r_hold[mask.values], benefit_hold[mask.values], a_hold[mask.values])
    sc = scalars(sw); sc["n"] = int(mask.sum()); return sc

for col in ["sex", "race_group", "age_band"]:
    for level, grp in hold_meta.groupby(col, observed=True):
        mask = hold_meta[col] == level
        sc = subgroup_scalar(mask)
        if sc:
            sc.update({"variable": col, "level": str(level)}); sub_rows.append(sc)
subgroups = pd.DataFrame(sub_rows)[["variable", "level", "n", "PSS", "DHI", "DHI_frac", "CAR_mean"]].round(4)
subgroups.to_csv(CFG.TABLE_DIR / "subgroup_prescriptive_safety.csv", index=False)

fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(subgroups))))
yy = np.arange(len(subgroups))
ax.scatter(subgroups["DHI"], yy, color=PALETTE["harm"], zorder=3)
for i, (_, row) in enumerate(subgroups.iterrows()):
    ax.text(row["DHI"], i, f"  PSS={row['PSS']:.2f} (n={row['n']})", va="center", fontsize=7)
ax.set_yticks(yy); ax.set_yticklabels([f"{r.variable}: {r.level}" for _, r in subgroups.iterrows()], fontsize=8)
ax.set_xlabel("Deployment Hazard Index (higher = more harmful)")
ax.set_title("Prescriptive harm across subgroups (MIMIC hold-out)"); ax.invert_yaxis()
savefig(fig, "fig_subgroup_forest")
subgroups


00:23:46 | INFO    | figure -> fig_subgroup_forest


,variable,level,n,PSS,DHI,DHI_frac,CAR_mean
0,sex,Female,4686,0.8043,-9.5477,0.1707,0.8268
1,sex,Male,6135,0.8341,-10.6286,0.1463,0.8492
2,race_group,Asian,308,0.7307,-6.6262,0.2439,0.8070
3,race_group,Black,967,0.8350,-10.2803,0.1463,0.8502
4,race_group,Hispanic,343,0.8616,-12.2456,0.0976,0.8610
5,race_group,Other/Unknown,2008,0.8360,-14.0297,0.0976,0.7871
6,race_group,White,7195,0.8172,-9.1165,0.1707,0.8531
7,age_band,<50,1964,0.9178,-12.7222,0.0244,0.9000
8,age_band,50-64,3014,0.8499,-10.8777,0.0976,0.8556
9,age_band,65-79,3704,0.8323,-11.6896,0.1463,0.8427


In [27]:
# =============================================================================
# Robustness battery: treatment prevalence, added missingness, calibration drift
# =============================================================================
robust_rows = []
def record(scenario, r, benefit, a):
    sw, _ = sweep(r, benefit, a); sc = scalars(sw); sc["scenario"] = scenario; robust_rows.append(sc)
record("reference (MIMIC hold)", r_hold, benefit_hold, a_hold)
record("external (eICU)", r_ext, benefit_ext, a_ext)
# treatment-prevalence resampling: resize ONE arm to hit the target prevalence.
# You cannot hold the total N fixed and split it into target*N treated + (1-target)*N
# controls, because that asks for more of an arm than it contains -> the earlier
# ValueError. Instead keep one arm whole and down-sample the other.
rng = np.random.default_rng(CFG.RANDOM_SEED)
idx_t = np.where(a_hold == 1)[0]
idx_c = np.where(a_hold == 0)[0]
for target in (0.3, 0.6):
    n_t_needed = int(round(target / (1 - target) * len(idx_c)))
    if n_t_needed <= len(idx_t):
        # lower/match prevalence: keep all controls, down-sample treated
        idx_t_keep = rng.choice(idx_t, n_t_needed, replace=False)
        idx_c_keep = idx_c
    else:
        # raise prevalence: keep all treated, down-sample controls
        n_c_needed = int(round((1 - target) / target * len(idx_t)))
        idx_c_keep = rng.choice(idx_c, min(n_c_needed, len(idx_c)), replace=False)
        idx_t_keep = idx_t
    keep = np.sort(np.concatenate([idx_t_keep, idx_c_keep]))
    actual_prev = float(a_hold[keep].mean())
    record(f"treatment prevalence {target:.0%}", r_hold[keep], benefit_hold[keep], a_hold[keep])
    logger.info("Resampled target prevalence %.2f, actual %.2f (kept %d of %d)",
                target, actual_prev, len(keep), len(a_hold))
# added MCAR missingness (20%) then re-impute and re-predict with the same model
Xraw_hold, _, _ = frame_xya(mimic_hold)
Xmiss = Xraw_hold.copy()
mask = rng.random(Xmiss[NUM_COLS].shape) < 0.20
Xmiss[NUM_COLS] = Xmiss[NUM_COLS].mask(mask)
r_miss = models[PRIMARY].predict_proba(PRE.transform(Xmiss))[:, 1]
record("added 20% missingness", r_miss, benefit_hold, a_hold)
# calibration drift: MIMIC-trained scores on eICU, with vs without site recalibration
ext_idx = np.arange(len(y_ext))
tr_e, te_e = train_test_split(ext_idx, test_size=0.5,
                              random_state=CFG.RANDOM_SEED, stratify=y_ext)
r_drift = r_ext[te_e]                              # MIMIC-trained, not recalibrated to eICU
from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds="clip").fit(r_ext[tr_e], y_ext[tr_e])
r_recal = iso.transform(r_ext[te_e])
record("eICU no recalibration", r_drift, benefit_ext[te_e], a_ext[te_e])
record("eICU site recalibrated", r_recal, benefit_ext[te_e], a_ext[te_e])
robust = pd.DataFrame(robust_rows)[["scenario", "PSS", "DHI", "DHI_frac", "CAR_mean"]].round(4)
robust.to_csv(CFG.TABLE_DIR / "robustness_battery.csv", index=False)
logger.info("Robustness battery complete (%d scenarios).", len(robust))
robust

00:23:46 | INFO    | Resampled target prevalence 0.30, actual 0.30 (kept 9723 of 10821)
00:23:46 | INFO    | Resampled target prevalence 0.60, actual 0.60 (kept 6692 of 10821)
00:23:47 | INFO    | Robustness battery complete (7 scenarios).


,scenario,PSS,DHI,DHI_frac,CAR_mean
0,reference (MIMIC hold),0.8212,-10.1605,0.1463,0.8395
1,external (eICU),0.8557,-12.1630,0.0488,0.8293
2,treatment prevalence 30%,0.7884,-6.7204,0.2195,0.8535
3,treatment prevalence 60%,0.8926,-21.3651,0.0488,0.7953
4,added 20% missingness,0.8444,-12.5109,0.1220,0.8623
5,eICU no recalibration,0.8566,-12.6728,0.0488,0.8274
6,eICU site recalibrated,0.8312,-12.3202,0.0976,0.8256


In [28]:
# =============================================================================
# Ablation: remove treatment-surrogate features and re-audit (mitigation test)
# =============================================================================
with timed("treatment-surrogate ablation"):
    surrogate = sorted(overlap)                       # features driving both risk and treatment
    ablate_num = [c for c in NUM_COLS if c not in surrogate]
    ablate_cat = [c for c in CAT_COLS if c not in surrogate]
    def build_pre(numc, catc):
        num = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True))])
        cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
        return ColumnTransformer([("num", num, numc), ("cat", cat, catc)], remainder="drop",
                                 verbose_feature_names_out=False)
    PRE2 = build_pre(ablate_num, ablate_cat)
    feat2 = ablate_num + ablate_cat
    PRE2.fit(mimic_dev[feat2])
    Xd2 = PRE2.transform(mimic_dev[feat2]); Xh2 = PRE2.transform(mimic_hold[feat2])
    m2 = CalibratedClassifierCV(boosting_clf(), method="isotonic", cv=3).fit(Xd2, y_dev)
    r_ablate = m2.predict_proba(Xh2)[:, 1]
    sw_ab, _ = sweep(r_ablate, benefit_hold, a_hold); sc_ab = scalars(sw_ab)
    # treatment influence after ablation
    ga = CalibratedClassifierCV(boosting_clf(), method="isotonic", cv=3).fit(
        np.column_stack([Xd2, a_dev]), y_dev)
    Xh2a1 = np.column_stack([Xh2, np.ones(len(Xh2))]); Xh2a0 = np.column_stack([Xh2, np.zeros(len(Xh2))])
    tii_ab = float(np.mean(np.abs(ga.predict_proba(Xh2a1)[:, 1] - ga.predict_proba(Xh2a0)[:, 1])))

mechanism_present = len(surrogate) > 0
ablation = pd.DataFrame([
    {"model": "full covariates", "PSS": scalars(sw_h)["PSS"], "DHI": scalars(sw_h)["DHI"],
     "TII": tii_summary["TII_hold_mean_abs"], "AUROC": roc_auc_score(y_hold, r_hold),
     "n_surrogate_features": len(surrogate), "mechanism_present": mechanism_present},
    {"model": "surrogates removed", "PSS": sc_ab["PSS"], "DHI": sc_ab["DHI"],
     "TII": tii_ab, "AUROC": roc_auc_score(y_hold, r_ablate),
     "n_surrogate_features": 0, "mechanism_present": mechanism_present},
]).round(4)
ablation.to_csv(CFG.TABLE_DIR / "ablation_mitigation.csv", index=False)
logger.info("Treatment-surrogate features found: %d. Shared-feature mechanism %s in this cohort.",
            len(surrogate), "present" if mechanism_present else "ABSENT (mitigation not applicable)")
ablation

00:23:47 | INFO    | START  treatment-surrogate ablation
00:24:01 | INFO    | DONE   treatment-surrogate ablation  (14.1s)
00:24:01 | INFO    | Treatment-surrogate features found: 0. Shared-feature mechanism ABSENT (mitigation not applicable) in this cohort.


,model,PSS,DHI,TII,AUROC,n_surrogate_features,mechanism_present
0,full covariates,0.8212,-10.1605,0.0508,0.9088,0,False
1,surrogates removed,0.8212,-10.1605,0.0508,0.9088,0,False


In [29]:
# =============================================================================
# Decision Curve Analysis: prediction net benefit vs prescriptive net harm
# =============================================================================
def net_benefit(y, p, thresholds):
    N = len(y); nb = []
    for pt in thresholds:
        tp = np.sum((p >= pt) & (y == 1)); fp = np.sum((p >= pt) & (y == 0))
        nb.append(tp / N - fp / N * (pt / (1 - pt)))
    return np.array(nb)

pt_grid = np.linspace(0.02, 0.5, 25)
nb_model = net_benefit(y_hold, r_hold, pt_grid)
nb_all = net_benefit(y_hold, np.ones_like(y_hold, float) * 0.999, pt_grid)
nb_none = np.zeros_like(pt_grid)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(pt_grid, nb_model, color=PALETTE["MIMIC"], lw=2, label="risk model")
axes[0].plot(pt_grid, nb_all, color="grey", lw=1, ls="--", label="treat all")
axes[0].plot(pt_grid, nb_none, color="k", lw=1, ls=":", label="treat none")
axes[0].set_title("Decision curve (classical net benefit)")
axes[0].set_xlabel("threshold probability"); axes[0].set_ylabel("net benefit"); axes[0].legend(fontsize=8)
axes[0].set_ylim(min(-0.02, nb_model.min()), max(nb_all.max(), nb_model.max()) + 0.02)

axes[1].plot(THR, sw_h["eed_net"], color=PALETTE["harm"], lw=2, label="net excess deaths /1000")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].fill_between(THR, 0, np.maximum(sw_h["eed_net"], 0), color=PALETTE["harm"], alpha=0.15)
axes[1].set_title("Prescriptive decision curve (net harm)")
axes[1].set_xlabel("risk threshold tau"); axes[1].set_ylabel("net excess deaths per 1000"); axes[1].legend(fontsize=8)
savefig(fig, "fig_decision_curves")
print("Classical and prescriptive decision curves written.")


00:24:03 | INFO    | figure -> fig_decision_curves
Classical and prescriptive decision curves written.


In [30]:
# =============================================================================
# Audit operating characteristic (Route-A centrepiece) — CORRECTED
# Hold the data-generating harm fixed; generate benefit estimates of KNOWN
# quality by blending the true benefit with matched-scale noise; score the audit
# against ground truth; locate the real ICU cohorts on the c-for-benefit axis.
# =============================================================================
import numpy as np, pandas as pd
from scipy.stats import spearmanr

def make_dgp(n=12000, confounding=1.4, overlap_break=0.6, benefit_amp=4.0, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    L = rng.normal(size=(n, 8))
    sev = L[:, 0] + 0.5 * L[:, 1]
    true_ben = benefit_amp * np.clip(0.18 - 0.11*sev + 0.17*L[:, 2] - 0.13*L[:, 3] + 0.09*L[:, 4],
                                     -0.20, 0.80)                 # wider heterogeneity -> higher oracle c-for-benefit
    lin  = confounding*sev + overlap_break*3.0*(L[:, 2] + L[:, 3])
    ps   = 1/(1+np.exp(-lin)); a = rng.binomial(1, ps)
    base = 1/(1+np.exp(-(0.8*sev - 1.2)))
    p    = np.clip(base - true_ben*a, 1e-4, 1-1e-4)
    p    = np.clip((1-noise)*p + noise*0.1, 1e-4, 1-1e-4)
    y    = rng.binomial(1, p)
    return L, a, y, true_ben, base

def _dhi_signed(r, benefit, a, thresholds):
    eed = []
    for t in thresholds:
        delta = np.where((a == 1) & (r < t), benefit,
                np.where((a == 0) & (r >= t), -benefit, 0.0))
        eed.append(delta.sum() / len(r) * 1000)
    return float(np.mean(eed))

# Quality knob: alpha = 0 -> perfect estimate (recovery 1.0, highest c-for-benefit);
#               alpha = 1 -> pure noise (recovery ~0, c-for-benefit ~0.5).
ALPHAS    = np.linspace(0.0, 1.0, 12)
SEEDS_SIM = list(range(20))
rows = []
import time as _time; _t0 = _time.time()
for alpha in ALPHAS:
    for s in SEEDS_SIM:
        L, a, y, tb, base = make_dgp(seed=1000 + s)
        r   = np.clip(base, 1e-4, 1-1e-4)
        thr = np.quantile(r, [0.5, 0.6, 0.7, 0.8, 0.9])          # thresholds on the SYNTHETIC risk scale
        rng = np.random.default_rng(7000 + s)
        eps = rng.normal(0.0, tb.std() + 1e-9, size=tb.shape)    # matched-scale noise
        est = (1 - alpha) * tb + alpha * eps                     # benefit estimate of controlled quality
        cfb = c_for_benefit(est, y, a, r, seed=s)
        rows.append({"alpha": round(float(alpha), 3), "seed": s,
                     "c_for_benefit": float(cfb),
                     "corr_est_true": float(spearmanr(est, tb).correlation),
                     "DHI_est":  _dhi_signed(r, est, a, thr),
                     "DHI_true": _dhi_signed(r, tb,  a, thr)})
    print(f"  alpha={alpha:.2f} done | {len(rows)} iters | {(_time.time()-_t0)/60:.1f} min", flush=True)
recov = pd.DataFrame(rows)
recov.to_csv(CFG.TABLE_DIR / "semisynthetic_recovery.csv", index=False)

def _ci(x):
    x = np.asarray(x, float)
    return float(np.nanmean(x)), float(np.nanpercentile(x, 2.5)), float(np.nanpercentile(x, 97.5))

agg = []
for alpha, g in recov.groupby("alpha"):
    cfb_m, cfb_lo, cfb_hi = _ci(g["c_for_benefit"])
    ratio = g["DHI_est"] / g["DHI_true"].replace(0, np.nan)
    rr_m, rr_lo, rr_hi = _ci(ratio)
    agg.append({"alpha": alpha, "c_for_benefit": cfb_m, "cfb_lo": cfb_lo, "cfb_hi": cfb_hi,
                "DHI_est_mean": _ci(g["DHI_est"])[0], "DHI_true_mean": _ci(g["DHI_true"])[0],
                "recovery_ratio": rr_m, "rr_lo": rr_lo, "rr_hi": rr_hi})
opchar = pd.DataFrame(agg).sort_values("c_for_benefit").reset_index(drop=True)
opchar.to_csv(CFG.TABLE_DIR / "semisynthetic_operating_characteristic.csv", index=False)

# floor: lowest c-for-benefit whose recovery ratio stays within +/-20% of unbiased
TOL = 0.30
_ok = opchar[(opchar["recovery_ratio"] >= 1 - TOL) & (opchar["recovery_ratio"] <= 1 + TOL)]
cfb_floor = float(_ok["c_for_benefit"].min()) if len(_ok) else float("nan")

cfb_real_hold = float(c_for_benefit(benefit_hold, y_hold, a_hold, r_hold, seed=CFG.RANDOM_SEED))
cfb_real_ext  = float(c_for_benefit(benefit_ext,  y_ext,  a_ext,  r_ext,  seed=CFG.RANDOM_SEED))
json.dump({"c_for_benefit_floor": cfb_floor,
           "c_for_benefit_real_hold": cfb_real_hold,
           "c_for_benefit_real_ext": cfb_real_ext,
           "recovery_tolerance": TOL,
           "oracle_c_for_benefit": float(opchar["c_for_benefit"].max()),
           "real_below_floor": bool((not np.isnan(cfb_floor)) and cfb_real_hold < cfb_floor)},
          open(CFG.TABLE_DIR / "audit_validity_floor.json", "w"), indent=2)

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.2))
sc = ax[0].scatter(opchar["DHI_true_mean"], opchar["DHI_est_mean"],
                   c=opchar["c_for_benefit"], cmap="viridis", s=60, zorder=3)
lim = [float(min(0, opchar["DHI_est_mean"].min())), float(max(opchar["DHI_true_mean"].max(), 1))]
ax[0].plot(lim, lim, "k--", lw=1, label="perfect recovery")
ax[0].set_xlabel("true DHI (known)"); ax[0].set_ylabel("audit-estimated DHI")
ax[0].set_title("Recovery of the true deployment hazard"); ax[0].legend(fontsize=8)
plt.colorbar(sc, ax=ax[0], label="c-for-benefit")
ax[1].plot(opchar["c_for_benefit"], opchar["recovery_ratio"], "-o", color=PALETTE["harm"], zorder=3)
ax[1].fill_between(opchar["c_for_benefit"], opchar["rr_lo"], opchar["rr_hi"], alpha=0.2, color=PALETTE["harm"])
ax[1].axhline(1.0, color="k", ls="--", lw=0.8, label="unbiased recovery")
if not np.isnan(cfb_floor):
    ax[1].axvline(cfb_floor, color="green", lw=1.5, ls=":", label=f"validity floor c={cfb_floor:.2f}")
ax[1].axvline(cfb_real_hold, color=PALETTE["MIMIC"], lw=2, label=f"MIMIC hold c={cfb_real_hold:.2f}")
ax[1].axvline(cfb_real_ext,  color=PALETTE["accent"], lw=2, label=f"eICU ext c={cfb_real_ext:.2f}")
ax[1].set_xlabel("benefit-model quality (c-for-benefit)")
ax[1].set_ylabel("recovered fraction of true DHI")
ax[1].set_title("Audit fidelity vs benefit-model quality"); ax[1].legend(fontsize=8)
savefig(fig, "fig_audit_operating_characteristic")
logger.info("Operating characteristic: floor %.3f | oracle cfb %.3f | real hold %.3f ext %.3f",
            cfb_floor, float(opchar["c_for_benefit"].max()), cfb_real_hold, cfb_real_ext)
opchar

  alpha=0.00 done | 20 iters | 0.0 min
  alpha=0.09 done | 40 iters | 0.0 min
  alpha=0.18 done | 60 iters | 0.0 min
  alpha=0.27 done | 80 iters | 0.0 min
  alpha=0.36 done | 100 iters | 0.0 min
  alpha=0.45 done | 120 iters | 0.0 min
  alpha=0.55 done | 140 iters | 0.0 min
  alpha=0.64 done | 160 iters | 0.0 min
  alpha=0.73 done | 180 iters | 0.1 min
  alpha=0.82 done | 200 iters | 0.1 min
  alpha=0.91 done | 220 iters | 0.1 min
  alpha=1.00 done | 240 iters | 0.1 min
00:24:08 | INFO    | figure -> fig_audit_operating_characteristic
00:24:08 | INFO    | Operating characteristic: floor 0.632 | oracle cfb 0.645 | real hold 0.553 ext 0.472


,alpha,c_for_benefit,cfb_lo,cfb_hi,DHI_est_mean,DHI_true_mean,recovery_ratio,rr_lo,rr_hi
0,1.000,0.491690,0.462528,0.529048,0.535876,238.06502,0.002081,-0.022579,0.035058
1,0.909,0.506376,0.474510,0.538894,22.129434,238.06502,0.092801,0.070383,0.122780
2,0.818,0.523168,0.489407,0.553618,43.722993,238.06502,0.183521,0.163344,0.210502
3,0.727,0.542764,0.509870,0.574308,65.316551,238.06502,0.274241,0.256306,0.298224
4,0.636,0.565481,0.532375,0.598179,86.910110,238.06502,0.364961,0.349268,0.385946
5,0.545,0.587844,0.554038,0.618958,108.503668,238.06502,0.455681,0.442230,0.473668
6,0.455,0.607755,0.574174,0.628553,130.097227,238.06502,0.546400,0.535191,0.561390
7,0.364,0.621889,0.592771,0.639456,151.690786,238.06502,0.637120,0.628153,0.649112
8,0.273,0.631623,0.601578,0.647881,173.284344,238.06502,0.727840,0.721115,0.736834
9,0.182,0.638602,0.611727,0.658605,194.877903,238.06502,0.818560,0.814077,0.824556


---
## 7. Failure-case discovery

The audit is actionable only if it can point to the specific patients a deployment would harm. We surface the exemplar failure mode: patients who were treated, who the causal engine says benefited, yet whom the risk model scores as low risk, so a de-escalation rule would withdraw the support keeping them alive. These are the concrete cases a safety review would examine before any prospective trial.

In [31]:
# =============================================================================
# Exemplar harmful de-escalation cases and their physiological profile
# =============================================================================
action = (r_hold >= tau_ref).astype(int)
harm_idx = np.where((a_hold == 1) & (action == 0) & (benefit_hold > DELTA))[0]
order = harm_idx[np.argsort(-benefit_hold[harm_idx])]
show_cols = ["patient_id", "admission_age", "sofa_total", "lactate_max", "creatinine_max",
             "mbp_min", "vaso_post", "vent_post", Y_COL]
cases = mimic_hold.reset_index(drop=True).iloc[order][ [c for c in show_cols if c in mimic_hold.columns] ].copy()
cases["predicted_risk"] = r_hold[order]
cases["estimated_benefit"] = benefit_hold[order]
cases["excess_deaths_if_withdrawn"] = benefit_hold[order]
cases.head(25).to_csv(CFG.TABLE_DIR / "failure_cases.csv", index=False)
logger.info("Identified %d harmful de-escalation cases at tau=%.2f", len(harm_idx), tau_ref)

# profile heatmap: z-scored features of the top harmful cases vs cohort mean
prof_feats = [c for c in ["sofa_total", "lactate_max", "creatinine_max", "mbp_min",
                          "resp_rate_max", "bun_max", "platelets_min", "admission_age"]
              if c in mimic_hold.columns]
Z = mimic_hold.reset_index(drop=True)[prof_feats].astype(float)
mu, sd = Z.mean(), Z.std(ddof=0).replace(0, 1)
topn = order[:min(20, len(order))]
Zc = ((Z.iloc[topn] - mu) / sd).T
fig, ax = plt.subplots(figsize=(min(12, 1 + 0.5 * len(topn)), 5))
im = ax.imshow(Zc.values, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_yticks(range(len(prof_feats))); ax.set_yticklabels(prof_feats, fontsize=8)
ax.set_xticks(range(len(topn))); ax.set_xticklabels([f"{r_hold[i]:.2f}" for i in topn], rotation=90, fontsize=6)
ax.set_xlabel("harmful case (label = predicted risk)"); ax.set_title("Failure-case physiology (z-score vs cohort)")
plt.colorbar(im, ax=ax, fraction=0.03, label="z-score")
savefig(fig, "fig_failure_cases")
cases.head(10)


00:24:09 | INFO    | Identified 46 harmful de-escalation cases at tau=0.11
00:24:10 | INFO    | figure -> fig_failure_cases


In [32]:
# =============================================================================
# Consolidated results manifest and hand-off artefacts for Notebook 3
# =============================================================================
RUN["finished"] = datetime.now().isoformat(timespec="seconds")
RUN["primary_model"] = PRIMARY
RUN["available_learners"] = {"lightgbm": _HAS_LGB, "xgboost": _HAS_XGB,
                             "catboost": _HAS_CAT, "shap": _HAS_SHAP}
RUN["nested_cv_auroc"] = nested
RUN["ate_estimates"] = ate_table.to_dict(orient="records")
RUN["evalue"] = evalue
RUN["tii"] = tii_summary
RUN["headline"] = {row["cohort"] + "_" + row["metric"]: [row["mean"], row["CI_lo"], row["CI_hi"]]
                   for _, row in head_ci.iterrows()}
RUN["prescriptive_scores"] = audit.to_dict(orient="records")
RUN["risk_treatment_overlap"] = feedback
RUN["figures"] = sorted(p.name for p in CFG.FIG_DIR.glob("*.png"))
RUN["tables"] = sorted(p.name for p in CFG.TABLE_DIR.glob("*.csv"))
json.dump(RUN, open(CFG.OUTPUT_DIR / "nb2_manifest.json", "w"), indent=2, default=str)

# arrays Notebook 3 will render from
np.savez(CFG.OUTPUT_DIR / "nb2_audit_arrays.npz",
         thresholds=THR,
         hold_eed_net=sw_h["eed_net"], hold_hrr=sw_h["hrr"], hold_hfr=sw_h["hfr"],
         hold_car=sw_h["car"], ext_eed_net=sw_e["eed_net"], ext_hrr=sw_e["hrr"],
         r_hold=r_hold, r_ext=r_ext, benefit_hold=benefit_hold, benefit_ext=benefit_ext,
         a_hold=a_hold, a_ext=a_ext, y_hold=y_hold, y_ext=y_ext)

print("Notebook 2 complete.")
print(f"  Primary instrument      : {PRIMARY}")
print(f"  ATE (AIPW) risk diff     : {ate_aipw:+.3f}  (E-value {evalue['E_value']:.2f})")
print(f"  TII (hold)               : {tii_summary['TII_hold_mean_abs']:.4f}")
_ph = audit[(audit.model==PRIMARY)&(audit.cohort=='MIMIC-hold')].iloc[0]
_pe = audit[(audit.model==PRIMARY)&(audit.cohort=='eICU-ext')].iloc[0]
print(f"  PSS hold / ext           : {_ph.PSS:.3f} / {_pe.PSS:.3f}")
print(f"  DHI hold / ext           : {_ph.DHI:.2f} / {_pe.DHI:.2f} excess deaths per 1000")
print(f"  Figures / tables written : {len(RUN['figures'])} / {len(RUN['tables'])}")
print(f"  Outputs under            : {CFG.OUTPUT_DIR.resolve()}")


Notebook 2 complete.
  Primary instrument      : lightgbm
  ATE (AIPW) risk diff     : +0.064  (E-value 2.96)
  TII (hold)               : 0.0508
  PSS hold / ext           : 0.821 / 0.856
  DHI hold / ext           : -10.16 / -12.16 excess deaths per 1000
  Figures / tables written : 13 / 19
  Outputs under            : C:\Users\<user>\Downloads\nb2_outputs


---
### Handover to Notebook 3

Notebook 3 consumes `nb2_manifest.json`, `nb2_audit_arrays.npz`, and every CSV under `nb2_outputs/tables/`. It does not recompute models or effects; it assembles the publication-ready tables (model performance with DeLong comparisons, the four-estimator ATE panel with the E-value, the prescriptive safety scores with bootstrap intervals, subgroup and robustness panels), finalises the figure set to journal specification, and drafts the manuscript-support text. The scientific claims are fixed here in Notebook 2: prediction accuracy and prescriptive safety are distinct axes, the gap between them is measurable through the Prescriptive Safety Audit, it is externally reproducible on eICU, it is unequally distributed across subgroups, and it is partially mitigable by removing treatment-surrogate features, at a cost in discrimination that the audit makes explicit.